In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import pyodbc
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
from functools import reduce
import numpy as np

In [2]:
# Функция для форматирования даты в строковый формат 'DD.MM.YYYY'
def format_date_column(df, date_column):
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce', dayfirst=True).dt.strftime('%d.%m.%Y')
    return df

def format_elapsed_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours)} часа(ов) {int(minutes)} минут(ы) {seconds:.2f} секунд"
# Функция для подключения к SQL Server с аутентификацией Windows
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    Engine = create_engine(connection_string)
    return Engine
# Функция для проверки, является ли файл скрытым (для Windows)
def is_hidden(file_path):
    try:
        # Получаем атрибуты файла
        file_attributes = os.stat(file_path).st_file_attributes
        # Проверяем, установлен ли флаг "скрытый"
        return file_attributes & 2 != 0  # 2 соответствует атрибуту "скрытый"
    except Exception:
        # Если возникла ошибка, считаем файл не скрытым
        return False
# Функция для обработки ошибок и замены их на null
def handle_errors(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
    return df

In [3]:
# FOLDER_PATH = os.path.normpath(r"!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
# FOLDER_PATH_FEATURES = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Taldykin\Дашбоард по рекламным кампаниям")
# FOLDER_PATH_FOR_DB = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Taldykin\Дашбоард по рекламным кампаниям")
# FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Taldykin")

FOLDER_PATH = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
FOLDER_PATH_FEATURES = r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям"
FOLDER_PATH_FOR_DB = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям")
FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ")

SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [4]:
# 4. Получить данные из файла "Затраты ВБ_2.xlsx"
try:
    print("Начинаем получать данные для Затрат ВБ...")
    start_time = time.time()  # Запускаем таймер
    file_path_expenses = os.path.join(FOLDER_PATH, "Затраты ВБ_2.xlsx")

    if os.path.exists(file_path_expenses):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Дата", "Артикул WB", "ID кампании", "Показы", "Клики",
            "Кол-во добавлений в корзину", "Заказы, шт",
            "Кол-во заказаных товаров, шт", "Заказов на сумму",
            "Расход, с НДС"
        ]

        # Переименование столбцов
        column_mapping = {
            "Показы": "Рекламные показы",
            "Клики": "Рекламные клики",
            "Кол-во добавлений в корзину": "Рекламные в корзину",
            "Заказы, шт": "Рекламные Заказы, шт",
            "Кол-во заказаных товаров, шт": "Рекламные заказаных товаров, шт",
            "Заказов на сумму": "Рекламные заказов на сумму",
            "Расход, с НДС": "Расход, руб"
        }

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул WB": str,
            "Рекламные показы": int,
            "Рекламные клики": int,
            "Рекламные в корзину": int,
            "Рекламные Заказы, шт": int,
            "Рекламные заказаных товаров, шт": int,
            "Рекламные заказов на сумму": int,
            "Расход, руб": int
        }

        # Чтение файла с указанием нужных столбцов
        df_expenses = pd.read_excel(file_path_expenses, sheet_name="Затраты ВБ", engine="calamine",
                                    usecols=columns_to_read)

        # Переименование столбцо
        df_expenses.rename(columns=column_mapping, inplace=True)

        # Форматирование даты
        df_expenses = format_date_column(df_expenses, 'Дата')
        
        # df_expenses = df_expenses.drop_duplicates(subset=['Артикул WB', 'Дата'], keep='first')
        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Затраты:")
        print(df_expenses.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Затрат ВБ успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Затраты ВБ_2.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Затрат ВБ: {e}")

Начинаем получать данные для Затрат ВБ...
Первые 5 строк таблицы Затраты:
         Дата  Артикул WB  ID кампании  Рекламные показы  \
0  05.02.2026   756298841     33597079               561   
1  05.02.2026   525274813     30480381              1396   
2  05.02.2026   526244996     30480248               335   
3  05.02.2026   525274809     30480211              1787   
4  05.02.2026   463046230     30480157               187   

   Рекламные заказов на сумму  Рекламные Заказы, шт  Рекламные клики  \
0                           0                     0                9   
1                           0                     0               26   
2                           0                     0                7   
3                           0                     0               38   
4                           0                     0                3   

   Рекламные в корзину  Рекламные заказаных товаров, шт  Расход, руб  
0                    0                                0      

In [5]:
df_expenses.columns

Index(['Дата', 'Артикул WB', 'ID кампании', 'Рекламные показы',
       'Рекламные заказов на сумму', 'Рекламные Заказы, шт', 'Рекламные клики',
       'Рекламные в корзину', 'Рекламные заказаных товаров, шт',
       'Расход, руб'],
      dtype='object')

In [6]:
# file_path_campaing = os.path.join(FOLDER_PATH, "Campaign_Info.xlsx")
# df_campaing = pd.read_excel(file_path_campaing, engine='openpyxl')

In [7]:
# df_campaing = df_campaing[['name', 'advertId', 'type']]
# df_campaing = df_campaing.rename(columns={'type':'Раздел'})
# df_campaing['Раздел'] = df_campaing['Раздел'].astype(str)
# type_dictionary = {"8":"Автоматическое", 
#                    "9":"Аукцион",
#                    "4":"Каталог",
#                    "5":"Карточка",
#                    "6":"Поиск"}

# df_campaing["Раздел"] = df_campaing["Раздел"].map(type_dictionary)
# df_campaing

In [8]:
df_expenses

,Дата,Артикул WB,ID кампании,Рекламные показы,Рекламные заказов на сумму,"Рекламные Заказы, шт",Рекламные клики,Рекламные в корзину,"Рекламные заказаных товаров, шт","Расход, руб"
0,05.02.2026,756298841,33597079,561,0,0,9,0,0,340.0
1,05.02.2026,525274813,30480381,1396,0,0,26,3,0,683.0
2,05.02.2026,526244996,30480248,335,0,0,7,2,0,26.0
3,05.02.2026,525274809,30480211,1787,0,0,38,0,0,254.0
4,05.02.2026,463046230,30480157,187,0,0,3,0,0,72.0
...,...,...,...,...,...,...,...,...,...,...
266303,15.12.2025,302367829,30487158,2331,2398,2,54,2,2,511.0
266304,15.12.2025,253009036,30487157,14657,54016,23,287,51,23,3379.0
266305,15.12.2025,253493188,30487147,4922,24294,6,145,18,6,2140.0
266306,15.12.2025,302367822,30487404,2843,0,0,25,1,0,1200.0


In [9]:
df_expenses[df_expenses['Дата'] == "05.11.2025"]['Расход, руб'].sum()

np.float64(0.0)

In [10]:
pattern = r"История-затрат-Все*\.xlsx"

folder = os.path.join(FOLDER_PATH, "Затраты", "Затраты ВБ")
pattern = re.compile(r"^История-затрат-Все.*\.xlsx$", re.IGNORECASE)

files = [e.name for e in os.scandir(folder) if e.is_file() and pattern.match(e.name)]
# Если нужны полные пути:
# files = [e.path for e in os.scandir(folder) if e.is_file() and pattern.match(e.name)]

# Если файлы не найдены, выбрасываем ошибку
if not files:
    raise FileNotFoundError("Файл История-затрат не найден")
# Находим последний файл по времени изменения
latest = max(files, key=lambda f: os.path.getmtime(os.path.join(os.path.join(FOLDER_PATH, "Затраты/Затраты ВБ"), f)))

# Путь к последнему найденному файлу
latest_file_path = os.path.join(os.path.join(FOLDER_PATH, "Затраты/Затраты ВБ"), latest)

df_costs_hist = pd.read_excel(latest_file_path, engine='calamine')
df_costs_hist = df_costs_hist[['Кампания', 'ID кампании', 'Раздел']]
df_costs_hist['Раздел'] = df_costs_hist['Раздел'].astype(str)
type_dictionary = {"Единая Ставка":"Автоматическое", 
                   "Ручная Ставка":"Аукцион"}

df_costs_hist["Раздел"] = df_costs_hist["Раздел"].map(type_dictionary)
df_costs_hist

,Кампания,ID кампании,Раздел
0,2025-11-30 W2174239 АВТО ОА»,31075594,Автоматическое
1,test M5258767 v3,33409487,Аукцион
2,2025-11-16 W6256320 АВТО ОА»,30498774,Автоматическое
3,2025-11-30 W2174239 АВТО ОА»,31075594,Автоматическое
4,2025-11-15 S4157114 АВТО ОА»,30427903,Автоматическое
...,...,...,...
418840,2025-11-16 S8855076 АВТО ОА»,30481860,Автоматическое
418841,2025-11-16 W5258324 АВТО ОА»,30498228,Автоматическое
418842,2025-11-16 W8359004 АВТО ОА»,30484278,Автоматическое
418843,2025-12-04 W7588377 АВТО ОА»,31259587,Автоматическое


In [11]:
# # оставим только нужные поля из справочника кампаний
# camp_ref = df_costs_hist[['ID кампании', 'Раздел']].copy()

# df = (
#     df_expenses
#     .merge(camp_ref, on='ID кампании', how='left')
# )

# # если встречаются пустые разделы, обозначим их как "Прочее" (не обязательно)
# df['Раздел'] = df['Раздел'].fillna('Прочее')

# # --- 2) подготовим списки ключей и метрик ---
# key_cols = ['Дата', 'Артикул WB']              # ключ группировки
# aux_cols = ['ID кампании', 'Раздел']           # служебные
# # метрики = все остальные столбцы
# metric_cols = [c for c in df.columns if c not in (set(key_cols) | set(aux_cols))]

# # убеждаемся, что метрики — числовые
# for c in metric_cols:
#     df[c] = pd.to_numeric(df[c], errors='coerce')

# # --- 3) агрегация по (Дата, Артикул WB, Раздел) ---
# agg = (
#     df.groupby(key_cols + ['Раздел'], as_index=False)[metric_cols]
#       .sum(min_count=1)  # если все NaN — оставит NaN
# )

# # --- 4) разворот "Раздел" в колонки с префиксами Раздел_Метрика ---
# wide = agg.pivot_table(
#     index=key_cols,
#     columns='Раздел',
#     values=metric_cols,
#     aggfunc='sum',
#     fill_value=0
# )

# # имена колонок будут мультииндексом (метрика, раздел) — сплющим в "Раздел_Метрика"
# wide.columns = [f"{sec}_{met}" for met, sec in wide.columns.to_flat_index()]
# wide = wide.reset_index()

# # перечень разделов, которые реально есть (для суммирования и порядка)
# sections = [s for s in df['Раздел'].dropna().unique()]

# # --- 5) итоговые столбцы без префиксов = сумма по всем разделам ---
# for met in metric_cols:
#     # сложим по всем найденным разделам (Автоматическое, Аукцион, Прочее — если есть)
#     to_sum = [f"{sec}_{met}" for sec in sections if f"{sec}_{met}" in wide.columns]
#     if not to_sum:
#         continue
#     wide[met] = wide[to_sum].sum(axis=1)

# # --- 6) Упорядочим колонки: ключи → (по разделам) → итоги ---
# ordered_cols = key_cols[:]
# # сначала по разделам (Автоматическое и Аукцион будут первыми, если есть)
# for sec in ['Автоматическое', 'Аукцион'] + [s for s in sections if s not in ['Автоматическое', 'Аукцион']]:
#     ordered_cols += [c for c in wide.columns if c.startswith(f"{sec}_")]

# # и в самом конце — итоговые метрики без префиксов
# ordered_cols += [c for c in metric_cols if c in wide.columns]

# wide = wide[ordered_cols].copy()

# # # Результат: wide — таблица вида
# # # Дата | Артикул WB | Автоматическое_Показы | Аукцион_Показы | ... | Автоматическое_Расход, руб | Аукцион_Расход, руб | ... | Расход, руб
# # wide.head()
# # --- 4.1) ID кампании по разделам: берём первый в каждой группе ---
# id_wide = (
#     df.groupby(key_cols + ['Раздел'], as_index=False)['ID кампании']
#       .first()  # первый попавшийся в рамках (Дата, Артикул, Раздел)
#       .pivot(index=key_cols, columns='Раздел', values='ID кампании')
#       .rename(columns=lambda sec: f"{sec}_ID кампании")
#       .reset_index()
# )

# # добавим ID-столбцы к нашей широкой таблице метрик
# wide = wide.merge(id_wide, on=key_cols, how='left')

# # --- 4.2) Тип активности по наличию кампаний в разделах ---
# auto_id_col = 'Автоматическое_ID кампании'
# auc_id_col  = 'Аукцион_ID кампании'

# # безопасно создаём булевы признаки наличия разделов (если колонки могут отсутствовать)
# auto_present = wide[auto_id_col].notna() if auto_id_col in wide.columns else pd.Series(False, index=wide.index)
# auc_present  = wide[auc_id_col].notna()  if auc_id_col  in wide.columns else pd.Series(False, index=wide.index)

# wide['Тип активности'] = np.select(
#     [
#         auto_present & auc_present,
#         auto_present,
#         auc_present,
#     ],
#     [
#         'Автоматическое и Аукцион',
#         'Автоматическое',
#         'Аукцион'
#     ],
#     default='Органика'
# )

# # --- 5) итоговые столбцы без префиксов = сумма по всем разделам (как раньше) ---
# for met in metric_cols:
#     to_sum = [f"{sec}_{met}" for sec in sections if f"{sec}_{met}" in wide.columns]
#     if to_sum:
#         wide[met] = wide[to_sum].sum(axis=1)

# # --- 6) Порядок колонок: ключи → Тип активности → ID по разделам → метрики по разделам → итоги ---
# section_order = ['Автоматическое', 'Аукцион'] + [s for s in sections if s not in ['Автоматическое', 'Аукцион']]

# ordered_cols = key_cols[:] + ['Тип активности']  # добавили сюда
# for sec in section_order:
#     id_col = f"{sec}_ID кампании"
#     if id_col in wide.columns:
#         ordered_cols.append(id_col)
#     ordered_cols += [c for c in wide.columns if c.startswith(f"{sec}_") and c != id_col]

# ordered_cols += [c for c in metric_cols if c in wide.columns]
# wide = wide[ordered_cols].copy()
# # --- 4.3) Бинарные признаки по типам активности ---
# wide['Автоматическое'] = auto_present.astype(int)
# wide['Аукцион'] = auc_present.astype(int)
# df_sum = wide

In [12]:
# Маппинг «сырых» значений разделов -> нормализованные
TYPE_DICT_RAW = {
    "Единая Ставка": "Автоматическое",
    "Ручная Ставка": "Аукцион"
}

# Приоритет при конфликте (к одному ID привязаны разные разделы)
SECTION_PRIORITY = {'Аукцион': 2, 'Автоматическое': 1, np.nan: 0}

# Как называть «неопределённый» раздел
MISC_SECTION_NAME = 'Прочее'

# Кандидаты для названия столбца расхода в df_expenses
SPEND_COL_CANDIDATES = ['Расход, ₽', 'Расход, руб', 'Расход, Р', 'Расход']

# Ключи в wide-таблице
KEY_COLS = ['Дата', 'Артикул WB']


# ===================== ВСПОМОГАТЕЛЬНЫЕ =====================

def _pick_spend_col(df: pd.DataFrame) -> str:
    for c in SPEND_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError("Не найдена колонка расхода среди: " + ", ".join(SPEND_COL_CANDIDATES))


def build_camp_ref(df_costs_hist: pd.DataFrame) -> pd.DataFrame:
    """
    На вход: df_costs_hist с колонками как минимум ['ID кампании', 'Раздел'].
    Возвращает camp_ref: по одному 'Раздел' на каждый 'ID кампании'.
    - Нормализует 'Раздел' через TYPE_DICT_RAW (прочее -> NaN)
    - При нескольких значениях для одного ID выбирает по SECTION_PRIORITY
    """
    if df_costs_hist.empty:
        return pd.DataFrame({'ID кампании': pd.Series(dtype='Int64'),
                             'Раздел': pd.Series(dtype='object')})

    dfp = df_costs_hist[['ID кампании', 'Раздел']].copy()
    dfp['ID кампании'] = pd.to_numeric(dfp['ID кампании'], errors='coerce').astype('Int64')

    # Нормализация разделов (если уже нормализованы, невидимые значения станут NaN — это ок)
    dfp['Раздел'] = dfp['Раздел'].map(TYPE_DICT_RAW).where(dfp['Раздел'].map(TYPE_DICT_RAW).notna(), dfp['Раздел'])

    # Приоритет и выбор одного значения на ID
    dfp['_prio'] = dfp['Раздел'].map(SECTION_PRIORITY).fillna(0).astype(int)
    dfp = dfp.sort_values(['ID кампании', '_prio'], ascending=[True, False])
    camp_ref = dfp.drop_duplicates(subset=['ID кампании'], keep='first')[['ID кампании', 'Раздел']].copy()

    # sanity: уникальность справа
    assert not camp_ref['ID кампании'].duplicated().any(), "camp_ref содержит дубликаты ID кампании"
    return camp_ref


def merge_expenses_with_sections(df_expenses: pd.DataFrame, camp_ref: pd.DataFrame) -> pd.DataFrame:
    """
    Левый merge строго m:1. Исключает размножение строк.
    """
    out = df_expenses.copy()
    out['ID кампании'] = pd.to_numeric(out['ID кампании'], errors='coerce').astype('Int64')

    # Строгая валидация 'm:1' – упадёт, если camp_ref не уникален по ID
    out = out.merge(camp_ref, on='ID кампании', how='left', validate='m:1')
    out['Раздел'] = out['Раздел'].fillna(MISC_SECTION_NAME)
    return out


import numpy as np
import pandas as pd

# ===================== ПАРАМЕТРЫ =====================

# Если в справочнике встречаются "Единая Ставка"/"Ручная Ставка" — нормализуем
TYPE_DICT_RAW = {
    "Единая Ставка": "Автоматическое",
    "Ручная Ставка": "Аукцион"
}

# При конфликте нескольких "Раздел" на один ID — выбираем по приоритету
SECTION_PRIORITY = {'Аукцион': 2, 'Автоматическое': 1, np.nan: 0}

# Как называть неопределённый раздел (после merge)
MISC_SECTION_NAME = 'Прочее'

# Кандидаты названия столбца расхода в df_expenses
SPEND_COL_CANDIDATES = ['Расход, ₽', 'Расход, руб', 'Расход, Р', 'Расход']

# Ключи формирования wide
KEY_COLS = ['Дата', 'Артикул WB']

# Разделы, которые считаем платными (для логической "Органики")
PAID_SECTIONS = ['Автоматическое', 'Аукцион']


# ===================== ВСПОМОГАТЕЛЬНЫЕ =====================

def _pick_spend_col(df: pd.DataFrame) -> str:
    for c in SPEND_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise KeyError("Не найдена колонка расхода среди: " + ", ".join(SPEND_COL_CANDIDATES))


def build_camp_ref(df_costs_hist: pd.DataFrame) -> pd.DataFrame:
    """
    На вход: df_costs_hist с колонками как минимум ['ID кампании','Раздел'].
    Возвращает camp_ref: по одному 'Раздел' на каждый 'ID кампании' (по приоритету).
    """
    if df_costs_hist.empty:
        return pd.DataFrame({'ID кампании': pd.Series(dtype='Int64'),
                             'Раздел': pd.Series(dtype='object')})

    dfp = df_costs_hist[['ID кампании', 'Раздел']].copy()
    dfp['ID кампании'] = pd.to_numeric(dfp['ID кампании'], errors='coerce').astype('Int64')

    # Нормализация: заменяем только известные сырьевые значения, остальное оставляем как есть
    dfp['Раздел'] = dfp['Раздел'].replace(TYPE_DICT_RAW)

    # Приоритет и выбор одного значения на ID
    dfp['_prio'] = dfp['Раздел'].map(SECTION_PRIORITY).fillna(0).astype(int)
    dfp = dfp.sort_values(['ID кампании', '_prio'], ascending=[True, False])
    camp_ref = dfp.drop_duplicates(subset=['ID кампании'], keep='first')[['ID кампании', 'Раздел']].copy()

    # sanity: справа строго 1 строка на ID
    assert not camp_ref['ID кампании'].duplicated().any(), "camp_ref содержит дубликаты ID кампании"
    return camp_ref


def merge_expenses_with_sections(df_expenses: pd.DataFrame, camp_ref: pd.DataFrame) -> pd.DataFrame:
    """
    Левый merge строго m:1. Исключает размножение строк (и, как следствие, раздутые суммы).
    """
    out = df_expenses.copy()
    out['ID кампании'] = pd.to_numeric(out['ID кампании'], errors='coerce').astype('Int64')
    out = out.merge(camp_ref, on='ID кампании', how='left', validate='m:1')
    out['Раздел'] = out['Раздел'].fillna(MISC_SECTION_NAME)
    return out


def build_wide_by_section(df_merged: pd.DataFrame, key_cols=KEY_COLS) -> pd.DataFrame:
    """
    Формирует wide-таблицу:
      - <Раздел>_<Метрика> по каждой метрике
      - Итоги (без префиксов) как сумма по всем разделам
      - ID по разделам, 'Тип активности'
      - Бинарные флаги по разделам + логическая 'Органика'
    """
    df = df_merged.copy()
    key_cols = list(key_cols)
    aux_cols = ['ID кампании', 'Раздел']
    metric_cols = [c for c in df.columns if c not in set(key_cols + aux_cols)]

    # Приведение метрик к числам
    for c in metric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # 1) Суммы по (Дата, Артикул WB, Раздел)
    agg = (df.groupby(key_cols + ['Раздел'], as_index=False)[metric_cols]
             .sum(min_count=1))

    # 2) Pivot → (метрика, раздел) → Раздел_Метрика
    wide = agg.pivot_table(index=key_cols, columns='Раздел', values=metric_cols,
                           aggfunc='sum', fill_value=0)
    wide.columns = [f"{sec}_{met}" for met, sec in wide.columns.to_flat_index()]
    wide = wide.reset_index()

    # Актуальные разделы
    sections = list(pd.unique(df['Раздел'].dropna()))

    # 3) Итоги без префиксов (ОДИН раз по каждой метрике)
    for met in metric_cols:
        cols_to_sum = [f"{sec}_{met}" for sec in sections if f"{sec}_{met}" in wide.columns]
        if cols_to_sum:
            wide[met] = wide[cols_to_sum].sum(axis=1)

    # 4) ID по разделам (первый попавшийся в группе)
    id_wide = (df.groupby(key_cols + ['Раздел'], as_index=False)['ID кампании']
                 .first()
                 .pivot(index=key_cols, columns='Раздел', values='ID кампании')
                 .rename(columns=lambda sec: f"{sec}_ID кампании")
                 .reset_index())
    wide = wide.merge(id_wide, on=key_cols, how='left')

    # 5) Тип активности (по наличию ID в платных разделах)
    presence = {}
    for sec in PAID_SECTIONS:
        col = f'{sec}_ID кампании'
        presence[sec] = wide[col].notna() if col in wide.columns else pd.Series(False, index=wide.index)

    if len(PAID_SECTIONS) >= 2:
        both = presence[PAID_SECTIONS[0]] & presence[PAID_SECTIONS[1]]
        only_first = presence[PAID_SECTIONS[0]] & ~presence[PAID_SECTIONS[1]]
        only_second = ~presence[PAID_SECTIONS[0]] & presence[PAID_SECTIONS[1]]
        wide['Тип активности'] = np.select(
            [both, only_first, only_second],
            [f'{PAID_SECTIONS[0]} и {PAID_SECTIONS[1]}', PAID_SECTIONS[0], PAID_SECTIONS[1]],
            default='Органика'
        )
    else:
        # если платный только один тип
        only = presence[PAID_SECTIONS[0]]
        wide['Тип активности'] = np.where(only, PAID_SECTIONS[0], 'Органика')

    # 6) БИНАРНЫЕ ФЛАГИ ПО РАЗДЕЛАМ (one-hot) + ЛОГИЧЕСКАЯ "ОРГАНИКА"
    flags_block = (
        df.assign(_hit=1)
          .pivot_table(index=key_cols, columns='Раздел', values='_hit',
                       aggfunc='max', fill_value=0)
          .astype('int8')
          .reset_index()
    )
    wide = wide.merge(flags_block, on=key_cols, how='left')

    # Гарантируем наличие флагов для платных разделов
    for sec in PAID_SECTIONS:
        if sec not in wide.columns:
            wide[sec] = 0

    # Логическая "Органика" = 1, если ни один платный раздел не активен
    wide['Органика'] = (1 - wide[PAID_SECTIONS].max(axis=1)).astype('int8')

    # 7) Порядок колонок
    section_order = PAID_SECTIONS + [s for s in sections if s not in PAID_SECTIONS]
    ordered = key_cols + ['Тип активности']

    # Бинарные флаги сначала: платные, затем прочие, затем "Органика"
    ordered += [s for s in PAID_SECTIONS if s in wide.columns]
    ordered += [s for s in sections if (s not in PAID_SECTIONS and s in wide.columns and s != 'Органика')]
    ordered += ['Органика'] if 'Органика' in wide.columns else []

    # ID и метрики по разделам → итоги
    for sec in section_order:
        id_col = f"{sec}_ID кампании"
        if id_col in wide.columns:
            ordered.append(id_col)
        ordered += [c for c in wide.columns if c.startswith(f"{sec}_") and c != id_col]
    ordered += [c for c in metric_cols if c in wide.columns]

    wide = wide[[c for c in ordered if c in wide.columns]].copy()
    return wide


def build_sum_table(df_expenses: pd.DataFrame, df_costs_hist: pd.DataFrame) -> pd.DataFrame:
    """
    End-to-end: camp_ref (one-row-per-ID) -> merge (m:1) -> wide (+флаги).
    Печатает контроль суммы расходов до/после.
    """
    spend_col = _pick_spend_col(df_expenses)
    base_sum = pd.to_numeric(df_expenses[spend_col], errors='coerce').sum()

    camp_ref = build_camp_ref(df_costs_hist)
    df_merged = merge_expenses_with_sections(df_expenses, camp_ref)
    df_sum = build_wide_by_section(df_merged, key_cols=KEY_COLS)

    if spend_col in df_sum.columns:
        after_sum = pd.to_numeric(df_sum[spend_col], errors='coerce').sum()
        print(f"[CHECK] Сумма расходов до/после: {base_sum:,.0f} → {after_sum:,.0f}")
        if not np.isclose(after_sum, base_sum, rtol=1e-3, atol=1e-2):
            print("[WARN] Расхождение суммы расходов. Проверьте конфликтные ID кампаний или строки без ID.")
    else:
        print(f"[INFO] В df_sum нет '{spend_col}' для проверки суммы.")

    return df_sum



def build_sum_table(df_expenses: pd.DataFrame, df_costs_hist: pd.DataFrame) -> pd.DataFrame:
    """
    End-to-end: camp_ref (one-row-per-ID) -> merge (m:1) -> wide.
    Печатает контроль суммы расходов до/после.
    """
    # Базовая сумма расходов до пайплайна
    spend_col = _pick_spend_col(df_expenses)
    base_sum = pd.to_numeric(df_expenses[spend_col], errors='coerce').sum()

    # camp_ref и merge
    camp_ref = build_camp_ref(df_costs_hist)
    df_merged = merge_expenses_with_sections(df_expenses, camp_ref)

    # wide
    df_sum = build_wide_by_section(df_merged, key_cols=KEY_COLS)

    # Контроль суммы расходов
    if spend_col in df_sum.columns:
        after_sum = pd.to_numeric(df_sum[spend_col], errors='coerce').sum()
        print(f"[CHECK] Сумма расходов до/после: {base_sum:,.0f} → {after_sum:,.0f}")
        if not np.isclose(after_sum, base_sum, rtol=1e-3, atol=1e-2):
            print("[WARN] Расхождение суммы расходов. Диагностика конфликтных ID:")
            bad = (df_costs_hist.groupby('ID кампании')['Раздел']
                   .nunique().sort_values(ascending=False))
            bad = bad[bad > 1]
            if not bad.empty:
                print("ID с несколькими значениями 'Раздел':")
                print(bad.head(20))
            else:
                null_id_sum = pd.to_numeric(df_expenses.loc[df_expenses['ID кампании'].isna(), spend_col], errors='coerce').sum()
                print(f"Сумма по строкам без ID кампании (ушли в '{MISC_SECTION_NAME}'): {null_id_sum:,.0f}")
    else:
        print(f"[INFO] В df_sum нет колонки '{spend_col}' для проверки суммы.")

    return df_sum


In [13]:
wide = build_sum_table(df_expenses=df_expenses, df_costs_hist=df_costs_hist)
wide

[CHECK] Сумма расходов до/после: 325,470,053 → 325,470,053


,Дата,Артикул WB,Тип активности,Автоматическое,Аукцион,Органика,Автоматическое_ID кампании,"Автоматическое_Расход, руб","Автоматическое_Рекламные Заказы, шт",Автоматическое_Рекламные в корзину,...,Аукцион_Рекламные заказов на сумму,Аукцион_Рекламные клики,Аукцион_Рекламные показы,Рекламные показы,Рекламные заказов на сумму,"Рекламные Заказы, шт",Рекламные клики,Рекламные в корзину,"Рекламные заказаных товаров, шт","Расход, руб"
0,01.01.2026,5259192,Автоматическое,1,0,0,31804656,118.0,1,2,...,0,0,0,1353,2296,1,32,2,1,118.0
1,01.01.2026,5259199,Автоматическое,1,0,0,31152108,122.0,2,13,...,0,0,0,1462,5500,2,70,13,2,122.0
2,01.01.2026,5473335,Автоматическое,1,0,0,30300233,1490.0,1,0,...,0,0,0,6410,4583,1,10,0,1,1490.0
3,01.01.2026,5594434,Автоматическое,1,0,0,31152082,49.0,2,3,...,0,0,0,632,1062,2,16,3,2,49.0
4,01.01.2026,5594442,Автоматическое,1,0,0,31203511,68.0,3,3,...,0,0,0,456,2809,3,14,3,3,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260332,31.12.2025,676230447,Автоматическое,1,0,0,32044223,1468.0,0,0,...,0,0,0,3372,0,0,31,0,0,1468.0
260333,31.12.2025,676230448,Автоматическое,1,0,0,32044225,715.0,0,1,...,0,0,0,1631,0,0,6,1,0,715.0
260334,31.12.2025,676230467,Автоматическое,1,0,0,31958628,624.0,0,0,...,0,0,0,1422,0,0,1,0,0,624.0
260335,31.12.2025,676230487,Автоматическое,1,0,0,31958627,672.0,0,0,...,0,0,0,1532,0,0,6,0,0,672.0


In [14]:
wide[wide['Дата'] == "05.11.2025"]['Расход, руб'].sum()

np.float64(0.0)

In [15]:
print(wide[wide['Тип активности'] == "Автоматическое"]['Расход, руб'].sum())
print(wide[wide['Тип активности'] == "Автоматическое"]['Автоматическое_Расход, руб'].sum())

271698065.0
271698065.0


In [16]:
print(wide[wide['Тип активности'] == "Автоматическое"]['Расход, руб'].sum())
print(wide[wide['Тип активности'] == "Аукцион"]['Расход, руб'].sum())
print(wide[wide['Тип активности'] == "Автоматическое и Аукцион"]['Расход, руб'].sum())
print(wide[wide['Тип активности'] == "Органика"]['Расход, руб'].sum())

271698065.0
45576266.0
8195722.0
0.0


In [17]:
print(wide['Расход, руб'].sum())

325470053.0


In [18]:
wide

,Дата,Артикул WB,Тип активности,Автоматическое,Аукцион,Органика,Автоматическое_ID кампании,"Автоматическое_Расход, руб","Автоматическое_Рекламные Заказы, шт",Автоматическое_Рекламные в корзину,...,Аукцион_Рекламные заказов на сумму,Аукцион_Рекламные клики,Аукцион_Рекламные показы,Рекламные показы,Рекламные заказов на сумму,"Рекламные Заказы, шт",Рекламные клики,Рекламные в корзину,"Рекламные заказаных товаров, шт","Расход, руб"
0,01.01.2026,5259192,Автоматическое,1,0,0,31804656,118.0,1,2,...,0,0,0,1353,2296,1,32,2,1,118.0
1,01.01.2026,5259199,Автоматическое,1,0,0,31152108,122.0,2,13,...,0,0,0,1462,5500,2,70,13,2,122.0
2,01.01.2026,5473335,Автоматическое,1,0,0,30300233,1490.0,1,0,...,0,0,0,6410,4583,1,10,0,1,1490.0
3,01.01.2026,5594434,Автоматическое,1,0,0,31152082,49.0,2,3,...,0,0,0,632,1062,2,16,3,2,49.0
4,01.01.2026,5594442,Автоматическое,1,0,0,31203511,68.0,3,3,...,0,0,0,456,2809,3,14,3,3,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260332,31.12.2025,676230447,Автоматическое,1,0,0,32044223,1468.0,0,0,...,0,0,0,3372,0,0,31,0,0,1468.0
260333,31.12.2025,676230448,Автоматическое,1,0,0,32044225,715.0,0,1,...,0,0,0,1631,0,0,6,1,0,715.0
260334,31.12.2025,676230467,Автоматическое,1,0,0,31958628,624.0,0,0,...,0,0,0,1422,0,0,1,0,0,624.0
260335,31.12.2025,676230487,Автоматическое,1,0,0,31958627,672.0,0,0,...,0,0,0,1532,0,0,6,0,0,672.0


In [19]:
df_expenses = wide

In [20]:
# # 1) Подготовим кампании: one-hot по "Раздел" (динамический набор столбцов)
# df_c = df_campaing[['advertId', 'Раздел']].dropna().copy()
# dummies = pd.get_dummies(df_c['Раздел'])          # столбцы = уникальные значения "Раздел"
# df_c = pd.concat([df_c[['advertId']], dummies], axis=1)

# # Если на один advertId есть несколько строк с разными "Раздел" — агрегируем флагами
# df_c = df_c.groupby('advertId', as_index=False).max()

# # 2) Присоединим к таблице с артикулами
# cols_campaign = dummies.columns.tolist()           # имена динамических столбцов
# df_result = (
#     df_expenses.merge(df_c, left_on='ID кампании', right_on='advertId', how='left')
#          .drop(columns=['advertId'])
# )

# # 3) Заполним пропуски нулями и приведём к int (0/1)
# if cols_campaign:  # на случай, если внезапно "Раздел" пустой
#     df_result[cols_campaign] = df_result[cols_campaign].fillna(0).astype(int)

# # Готово: df_result имеет вид
# # АртикулWB | ID кампании | Автоматическое | Аукцион | Каталог | Карточка | Поиск
# categories = df_campaing['Раздел'].unique().tolist()

# # Список датафреймов по категориям
# dfs_by_cat = []

# df_expenses = df_result
# display(df_expenses)

# for cat in categories:
#     if cat in df_result.columns:   # проверим, что колонка существует
#         df_group = df_result[df_result[cat] == 1].copy()
#         dfs_by_cat.append(df_group)

# # --- 0. Подготовка one-hot по Раздел (как у тебя) ---
# df_c = df_campaing[['advertId', 'Раздел']].dropna().copy()
# dummies = pd.get_dummies(df_c['Раздел'])
# df_c = pd.concat([df_c[['advertId']], dummies], axis=1).groupby('advertId', as_index=False).max()

# cols_campaign = dummies.columns.tolist()
# df_result = (
#     df_expenses.merge(df_c, left_on='ID кампании', right_on='advertId', how='left')
#                .drop(columns=['advertId'])
# )
# if cols_campaign:
#     df_result[cols_campaign] = df_result[cols_campaign].fillna(0).astype(int)

# # --- 1) Режем по категориям ---
# categories = df_campaing['Раздел'].dropna().unique().tolist()
# dfs_by_cat = []
# for cat in categories:
#     if cat in df_result.columns:
#         dfs_by_cat.append(df_result.loc[df_result[cat] == 1].copy())

In [21]:
# 3. Получить данные таблицы с SQL (Цены)
try:
    print("Начинаем получать данные для Цен...")
    start_time = time.time()  # Запускаем таймер
    Engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)
    query_price = f"""
        SELECT DT AS [Дата], ITEMID AS [Артикул], AVG(SITE_PRICE) AS [Цена]
        FROM [DBPartners].[dbo].[WblmRepPriceDiscountWbReport]
        WHERE dt >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY DT, ITEMID
        ORDER BY DT
        DESC
    """
    df_price = pd.read_sql(query_price, Engine)
    df_price = format_date_column(df_price, 'Дата')

    # Вывод первых 5 строк

    print("Первые 5 строк таблицы Цена:")
    print(df_price.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Цен успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Цен: {e}")

Начинаем получать данные для Цен...
Первые 5 строк таблицы Цена:
         Дата   Артикул    Цена
0  05.02.2026  10507420   598.0
1  05.02.2026  W1200989  5749.0
2  05.02.2026  S8251719  1389.0
3  05.02.2026  02105290   199.0
4  05.02.2026  u18010Y0   712.0
Данные для Цен успешно сохранены. Время выполнения: 0 часа(ов) 2 минут(ы) 19.95 секунд


In [22]:
# df_price.memory_usage(deep=True).sum() / 1024**2

In [23]:
# 6. Получить данные из файла "Справочник.xlsx"
try:
    print("Начинаем получать данные для Справочника...")
    start_time = time.time()  # Запускаем таймер
    file_path_reference = os.path.join(FOLDER_PATH, "Справочник.xlsx")

    if os.path.exists(file_path_reference):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Артикул", "Наименование", "Коллекция",
            "Бренд", "Размер", "Сезон", "Направление", "Розничный отдел",
            "Модель", "Группа", "Бизнес-группа", "Техсегмент",
            "Байер", "Две последние коллекции", "Основной артикул",
            "Артикул WB", "Себестоимость с НДС",
            "Процент выкупа ВБ", "НДС", "Ответственный за группу", "Группа для отчетов"
        ]

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул": str,
            "Наименование": str,
            "Коллекция": str,
            "Размер": str,
            "Бренд": str,
            "Сезон": str,
            "Направление": str,
            "Розничный отдел": str,
            "Модель": str,
            "Группа": str,
            "Бизнес-группа": str,
            "Техсегмент": str,
            "Байер": str,
            "Две последние коллекции": str,
            "Основной артикул": str,
            "Артикул WB": str,
            "Себестоимость с НДС": float,
            "Процент выкупа ВБ": float,
            "НДС": int,
            "Ответственный за группу": str,
            "Группа для отчетов":str
        }

        # Чтение файла с указанием нужных столбцов и типов данных
        df_reference = pd.read_excel(
            file_path_reference,
            sheet_name="Выгрузка для справочника",
            engine="openpyxl",
            usecols=columns_to_read,
            dtype=column_dtypes
        )

        # Удаление дубликатов
        df_reference = df_reference.drop_duplicates()

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Справочник:")
        print(df_reference.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Справочника успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Справочник.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Справочника: {e}")

Начинаем получать данные для Справочника...
Первые 5 строк таблицы Справочник:
    Артикул                         Наименование Размер Коллекция Бренд Сезон  \
0  W9009967  Полуботинки женские зимние ZL25AW-5     38    2025AW  kari  зима   
1  W9009967  Полуботинки женские зимние ZL25AW-5     36    2025AW  kari  зима   
2  W9009967  Полуботинки женские зимние ZL25AW-5     40    2025AW  kari  зима   
3  W9009967  Полуботинки женские зимние ZL25AW-5     37    2025AW  kari  зима   
4  W9009967  Полуботинки женские зимние ZL25AW-5     41    2025AW  kari  зима   

     Направление Розничный отдел    Модель Бизнес-группа  ... Техсегмент  \
0  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
1  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
2  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
3  Женская обувь   Женская обувь  ZL25AW-5         Обувь  ...   flat (L)   
4  Женская обувь   Женская обувь  ZL25AW-5         Обу

In [24]:
# 5. Получить данные из файлов вложенной папки "Показатели по неделям ВБ"
try:
    print("Начинаем получать данные для Воронки...")
    start_time = time.time()  # Запускаем таймер
    folder_path_weeks = os.path.join(FOLDER_PATH, "Показатели по неделям ВБ")
    df_funnel = pd.DataFrame()

    if os.path.exists(folder_path_weeks):
        for file in os.listdir(folder_path_weeks):
            file_path = os.path.join(folder_path_weeks, file)

            # Пропускаем скрытые файлы
            if is_hidden(file_path):
                print(f"Пропущен скрытый файл: {file}")
                continue

            # Проверяем расширение файла
            if file.endswith((".xlsx", ".xls")):
                try:
                    # Список всех возможных столбцов
                    all_columns = [
                        "Номенклатура", "Рейтинг карточки", "Показы", "Переходы в карточку",
                        "Положили в корзину", "Заказали, шт", "Выкупили, шт",
                        "Отменили, шт", "Заказали на сумму, руб", "Выкупили на сумму, руб",
                        "Отменили на сумму, руб", "Средняя цена, руб", "Дата", "Рейтинг по отзывам"
                    ]

                    if file.endswith(".xlsx"):
                        temp_df = pd.read_excel(file_path, sheet_name="воронка ОЗОН", engine="calamine")
                    elif file.endswith(".xls"):
                        temp_df = pd.read_excel(file_path, sheet_name="воронка ОЗОН", engine="xlrd")

                    # Добавляем отсутствующие столбцы со значением None
                    for col in all_columns:
                        if col not in temp_df.columns:
                            # если числовой столбец — проставляем 0, иначе None
                            if col in ["Показы", "Показы на карточке товара", "Положили в корзину", 
                                    "Заказали, шт", "Выкупили, шт", "Отменили, шт"]:
                                temp_df[col] = 0
                            else:
                                temp_df[col] = None

                    # Выбираем только нужные столбцы
                    temp_df = temp_df[all_columns]

                    # Переименование столбцов
                    temp_df.rename(columns={
                        "Номенклатура": "Артикул WB",
                        "Рейтинг карточки": "Рейтинг карточки",
                        "Переходы в карточку": "Показы на карточке товара",
                        "Положили в корзину": "Положили в корзину",
                        "Заказали, шт": "Заказали, шт",
                        "Выкупили, шт": "Выкупили, шт",
                        "Отменили, шт": "Отменили, шт",
                        "Заказали на сумму, руб": "Заказали на сумму, руб",
                        "Выкупили на сумму, руб": "Выкупили на сумму, руб",
                        "Отменили на сумму, руб": "Отменили на сумму, руб",
                        "Средняя цена, руб": "Средняя цена, руб",
                        "Рейтинг по отзывам": "Рейтинг по отзывам"  # Новый столбец
                    }, inplace=True)

                    # Типы данных для столбцов
                    column_dtypes = {
                        "Артикул WB": str,
                        "Рейтинг карточки": float,  # Рейтинг может быть дробным числом
                        "Показы": int, 
                        "Показы на карточке товара": int,
                        "Положили в корзину": int,
                        "Заказали, шт": int,
                        "Выкупили, шт": int,
                        "Отменили, шт": int,
                        "Заказали на сумму, руб": float,
                        "Выкупили на сумму, руб": float,
                        "Отменили на сумму, руб": float,
                        "Средняя цена, руб": float,
                        "Рейтинг по отзывам": float  # Рейтинг по отзывам также может быть дробным
                    }

                    # Применяем типы данных
                    for col, dtype in column_dtypes.items():
                        if col in temp_df.columns:
                            temp_df[col] = temp_df[col].astype(dtype, errors='ignore')

                    # Форматирование даты
                    temp_df = format_date_column(temp_df, 'Дата')

                    # Объединяем временный DataFrame с основным
                    df_funnel = pd.concat([df_funnel, temp_df], ignore_index=True)

                except Exception as e:
                    print(f"Ошибка при чтении файла {file}: {e}")

        # Выбираем финальные столбцы
        final_columns = [
            "Дата", "Артикул WB", "Рейтинг карточки", "Показы", "Показы на карточке товара",
            "Положили в корзину", "Заказали, шт", "Выкупили, шт",
            "Отменили, шт", "Заказали на сумму, руб", "Выкупили на сумму, руб",
            "Отменили на сумму, руб", "Средняя цена, руб", "Рейтинг по отзывам"
        ]
        df_funnel = df_funnel[final_columns]

        # Заполняем пустые значения в столбце "Рейтинг по отзывам" если они есть
        df_funnel["Рейтинг по отзывам"] = df_funnel["Рейтинг по отзывам"].fillna(0)

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Воронка:")
        print(df_funnel.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Воронки успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Папка 'Показатели по неделям ВБ' не найдена.")
except Exception as e:
    print(f"Ошибка при получении данных для Воронки: {e}")

Начинаем получать данные для Воронки...
Первые 5 строк таблицы Воронка:
         Дата Артикул WB  Рейтинг карточки  Показы  Показы на карточке товара  \
0  15.12.2025    8204399               8.5  335938                      22004   
1  15.12.2025  243289986               8.5  261086                      10944   
2  15.12.2025  174535735              10.0   93133                       6608   
3  15.12.2025  174533942               9.5   55454                       3648   
4  15.12.2025  243289533               8.5  172755                       8992   

   Положили в корзину  Заказали, шт  Выкупили, шт  Отменили, шт  \
0                1122           452             2             0   
1                 793           192             0             0   
2                 582           162             0             0   
3                 389           154             0             0   
4                 584           151             0             0   

   Заказали на сумму, руб  Выкупили на

In [25]:
# from functools import reduce
# from pandas.api.types import is_numeric_dtype

# # --- 2) Мёрджим датафреймы по ключам БЕЗ "ID кампании"
# #     Чтобы в итоге была ОДНА строка на пару (Дата, Артикул WB)
# base_keys = ['Дата', 'Артикул WB'] + cols_campaign   # <- это важно
# base_keys = [k for k in base_keys if k in df_result.columns]

# # НЕ префиксуем только base_keys
# def prefix_non_keys(df, prefix, keep_cols):
#     keep = set(keep_cols)
#     return df.rename(columns={c: f"{prefix}_{c}" for c in df.columns if c not in keep})

# dfs_prefixed = []
# for cat, df_cat in zip(categories, dfs_by_cat):
#     dfs_prefixed.append(prefix_non_keys(df_cat, cat, keep_cols=base_keys))

# # outer-merge всех категорий по (Дата, Артикул WB) → одна строка на эту пару
# df_final = reduce(lambda L, R: pd.merge(L, R, on=base_keys, how='outer'), dfs_prefixed)

In [26]:
# from pandas.api.types import is_numeric_dtype
# # def add_metric_totals_safe(df, decimal_comma=False):
# #     df = df.copy()
# #     pattern = re.compile(r'(.+?)_(.+)')   # "<категория>_<метрика>"
# #     col_map = {}                          # {метрика: [список колонок]}

# #     # Соберём группы колонок по метрикам
# #     for col in df.columns:
# #         m = pattern.fullmatch(col)
# #         if m:
# #             metric = m.group(2)
# #             col_map.setdefault(metric, []).append(col)

# #     def is_numeric_like(s: pd.Series) -> bool:
# #         s = s.dropna()
# #         if s.empty:
# #             return True
# #         if is_numeric_dtype(s):
# #             return True
# #         if s.dtype == 'object':
# #             x = s.astype(str)
# #             if decimal_comma:
# #                 x = x.str.replace(',', '.', regex=False)
# #             conv = pd.to_numeric(x, errors='coerce')
# #             return conv.notna().all()   # если есть нечисловые строки → False
# #         return False

# #     for metric, cols in col_map.items():
# #         # Если ХОТЬ В ОДНОМ столбце метрики есть текст — пропускаем всю метрику
# #         if not all(is_numeric_like(df[c]) for c in cols):
# #             continue

# #         # Конвертируем в числа и суммируем
# #         num_df = pd.DataFrame(index=df.index)
# #         for c in cols:
# #             s = df[c]
# #             if is_numeric_dtype(s):
# #                 num_df[c] = s
# #             else:
# #                 x = s.astype(str)
# #                 if decimal_comma:
# #                     x = x.str.replace(',', '.', regex=False)
# #                 num_df[c] = pd.to_numeric(x, errors='coerce')

# #         total_col = f"{metric}"
# #         df[total_col] = num_df.sum(axis=1, skipna=True)

# #         # Опционально: если без дробей — сделать int
# #         if pd.api.types.is_float_dtype(df[total_col]):
# #             s = df[total_col]
# #             if s.dropna().apply(float.is_integer).all():
# #                 df[total_col] = s.astype('int64')

# #     return df

# # --- 3) Суммы по метрикам (складываем все "<категория>_<метрика>" в столбец "<метрика>")
# def add_metric_totals_safe(df, decimal_comma=False):
#     df = df.copy()
#     pattern = re.compile(r'(.+?)_(.+)')   # "<категория>_<метрика>"
#     col_map = {}                          # {метрика: [список колонок]}

#     for col in df.columns:
#         m = pattern.fullmatch(col)
#         if m:
#             metric = m.group(2)
#             col_map.setdefault(metric, []).append(col)

#     def is_numeric_like(s: pd.Series) -> bool:
#         s = s.dropna()
#         if s.empty: return True
#         if is_numeric_dtype(s): return True
#         if s.dtype == 'object':
#             x = s.astype(str)
#             if decimal_comma: x = x.str.replace(',', '.', regex=False)
#             conv = pd.to_numeric(x, errors='coerce')
#             return conv.notna().all()
#         return False

#     for metric, cols in col_map.items():
#         # если среди колонок метрики встретится текст — пропускаем метрику
#         if not all(is_numeric_like(df[c]) for c in cols):
#             continue

#         num_df = pd.DataFrame(index=df.index)
#         for c in cols:
#             s = df[c]
#             if is_numeric_dtype(s):
#                 num_df[c] = s
#             else:
#                 x = s.astype(str)
#                 if decimal_comma: x = x.str.replace(',', '.', regex=False)
#                 num_df[c] = pd.to_numeric(x, errors='coerce')

#         df[metric] = num_df.sum(axis=1, skipna=True)  # итог без префикса

#         # красивый int, если без дробей
#         if pd.api.types.is_float_dtype(df[metric]):
#             s = df[metric]
#             if s.dropna().apply(float.is_integer).all():
#                 df[metric] = s.astype('int64')
#     return df

# df_sum = add_metric_totals_safe(df_final)

# # --- 4) Убираем все префиксные столбцы, оставляя только:
# # base_keys, ID по категориям (если нужны), флаги (если нужны) и ИТОГОВЫЕ метрики без префикса
# prefixed_cols = [c for c in df_sum.columns if any(c.startswith(f"{cat}_") for cat in categories)]

# # Если нужно СОХРАНИТЬ только категорийные ID-столбцы, отфильтруем их из удаления:
# keep_categorical_ids = [c for c in df_sum.columns if any(
#     c == f"{cat}_ID кампании" for cat in categories)]
# prefixed_cols_to_drop = [c for c in prefixed_cols if c not in keep_categorical_ids]

# df_sum = df_sum.drop(columns=prefixed_cols_to_drop)

# # (опционально) порядок колонок: ключи → категорийные ID → итоговые метрики
# ordered = base_keys + keep_categorical_ids + \
#           [c for c in df_sum.columns if c not in base_keys + keep_categorical_ids]
# df_sum = df_sum.reindex(columns=ordered)

In [28]:
df_funnel

,Дата,Артикул WB,Рейтинг карточки,Показы,Показы на карточке товара,Положили в корзину,"Заказали, шт","Выкупили, шт","Отменили, шт","Заказали на сумму, руб","Выкупили на сумму, руб","Отменили на сумму, руб","Средняя цена, руб",Рейтинг по отзывам
0,15.12.2025,8204399,8.5,335938,22004,1122,452,2,0,505303.0,2207.0,0.0,1118.0,4.8
1,15.12.2025,243289986,8.5,261086,10944,793,192,0,0,376707.0,0.0,0.0,1962.0,4.7
2,15.12.2025,174535735,10.0,93133,6608,582,162,0,0,443231.0,0.0,0.0,2736.0,4.8
3,15.12.2025,174533942,9.5,55454,3648,389,154,0,0,491914.0,0.0,0.0,3194.0,4.9
4,15.12.2025,243289533,8.5,172755,8992,584,151,0,0,334614.0,0.0,0.0,2216.0,4.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3255430,04.02.2026,804101953,9.0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0
3255431,04.02.2026,804102017,9.0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0
3255432,04.02.2026,804287755,10.0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0
3255433,04.02.2026,804316795,10.0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0


In [29]:
# 13. Связать "Воронка" с "Справочник"
try:
    print("Начинаем создавать таблицу ВоронкаСправочник...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Дата", "Артикул WB"]
    for col in required_columns:
        if col not in df_funnel.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_funnel.")
            exit()

    # Список столбцов, которые нужно взять из справочника
    reference_columns = [
        "Артикул", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
        "Розничный отдел", "Модель", "Группа", "Бизнес-группа", "Техсегмент",
        "Байер", "Две последние коллекции", "Основной артикул", "Артикул WB",
        "Себестоимость с НДС", "Процент выкупа ВБ", "НДС", "Ответственный за группу", "Группа для отчетов"
    ]

    # Фильтруем справочник, оставляя только нужные столбцы
    df_reference_filtered = df_reference[reference_columns]

    # Приводим типы данных к строковому формату
    df_funnel["Артикул WB"] = df_funnel["Артикул WB"].fillna('').astype(str)  # Заменяем NaN на пустые строки
    df_reference_filtered["Артикул WB"] = df_reference_filtered["Артикул WB"].fillna('').astype(str)

    # Объединение таблиц
    df_funnel_reference = pd.merge(
        df_funnel,
        df_reference_filtered,
        left_on="Артикул WB",
        right_on="Артикул WB",
        how="left"
    )

    # Удаление дубликатов
    df_funnel_reference = df_funnel_reference.drop_duplicates()

    # Удаление лишних столбцов (если они остались)
    # df_funnel_reference = df_funnel_reference.drop(columns=["Артикул WB"], errors="ignore")

    # Форматирование даты
    df_funnel_reference = format_date_column(df_funnel_reference, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВоронкаСправочник:")
    print(df_funnel_reference.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВоронкаСправочник успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочник: {e}")

# 14. Связать "ВоронкаСправочник" с "Затраты ВБ"
# try:
#     print("Начинаем создавать таблицу ВоронкаСправочникЗатраты...")

#     # Проверка наличия необходимых столбцов
#     required_columns = ["Дата", "Артикул WB"]
#     for col in required_columns:
#         if col not in df_funnel_reference.columns:
#             raise ValueError(f"Ошибка: Отсутствует столбец '{col}' в df_funnel_reference.")
#         if col not in df_expenses.columns:
#             raise ValueError(f"Ошибка: Отсутствует столбец '{col}' в df_expenses.")

#     # Очистка и нормализация данных
#     df_funnel_reference["Артикул WB"] = (
#         df_funnel_reference["Артикул WB"]
#         .fillna('')
#         .astype(str)
#         .str.strip()
#         .str.upper()
#     )
#     df_expenses["Артикул WB"] = (
#         df_expenses["Артикул WB"]
#         .fillna('')
#         .astype(str)
#         .str.strip()
#         .str.upper()
#     )

#     # Объединение таблиц
#     df_funnel_expenses = pd.merge(
#         df_funnel_reference,
#         df_expenses,
#         left_on=["Дата", "Артикул WB"],
#         right_on=["Дата", "Артикул WB"],
#         how="left",
#         indicator=True  # Для диагностики
#     )

#     # Проверка результата объединения
#     print("Результат объединения:")
#     print(df_funnel_expenses['_merge'].value_counts())

#     # Удаление лишних столбцов
#     df_funnel_expenses = df_funnel_expenses.drop(columns=["_merge"], errors="ignore")

#     # Форматирование даты
#     df_funnel_expenses = format_date_column(df_funnel_expenses, 'Дата')

#     # Вывод первых 5 строк
#     print("Первые 5 строк таблицы ВоронкаСправочникЗатраты:")
#     print(df_funnel_expenses.head())

#     print("Таблица ВоронкаСправочникЗатраты успешно создана.")

# except Exception as e:
#     print(f"Ошибка при создании таблицы ВоронкаСправочникЗатраты: {e}")

# ----------- настройки -----------
KEY_COLS = ['Дата', 'Артикул WB']
SPEND_CANDS = ['Расход, руб']  # названия возможного столбца расхода
ALLOC_MODE = 'even'   # 'first' — записать расход в первую строку группы; 'even' — равномерно разделить
NO_SUM_CONTAINS = ['ID кампании']  # числовые техполя, которые НЕЛЬЗЯ суммировать при агрегации справа

# ----------- утилиты -----------
def _pick_spend_col(df: pd.DataFrame) -> str:
    for c in SPEND_CANDS:
        if c in df.columns:
            return c
    raise KeyError(f"Не найден столбец расхода среди: {SPEND_CANDS}")

def _clean_money_series(s: pd.Series) -> pd.Series:
    # убираем пробелы/неразрывные и заменяем запятую на точку
    if s.dtype.kind in 'biufc':  # уже число
        return s
    s = (s.astype(str)
           .str.replace('\u00A0', '', regex=False)  # NBSP
           .str.replace(' ', '', regex=False)
           .str.replace(',', '.', regex=False))
    # убираем всё, что не цифра/точка/минус
    s = s.str.replace(r'[^0-9\.\-]+', '', regex=True)
    return pd.to_numeric(s, errors='coerce')

def _norm_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # дата: сначала пытаемся dayfirst, затем обычный парс; оставляем дату без времени
    # d1 = pd.to_datetime(out['Дата'], errors='coerce', dayfirst=True)
    # d2 = pd.to_datetime(out['Дата'], errors='coerce', dayfirst=False)
    # out['Дата'] = d1.fillna(d2).dt.normalize()
    # артикул
    out['Артикул WB'] = (out['Артикул WB']
                         .fillna('').astype(str).str.strip().str.upper())
    return out

def _make_right_unique(df_expenses: pd.DataFrame) -> pd.DataFrame:
    """
    Агрегируем df_expenses до 1 строки на (Дата, Артикул WB):
    - числовые метрики суммируем (кроме тех, чьё имя содержит токены из NO_SUM_CONTAINS);
    - бинарные флаги ('Автоматическое','Аукцион','Органика') берём max;
    - прочие текстовые — first.
    """
    df = _norm_keys(df_expenses).copy()

    # приведение расхода к числу и выбор его столбца
    spend_col = _pick_spend_col(df)
    df[spend_col] = _clean_money_series(df[spend_col])

    key = KEY_COLS
    other = [c for c in df.columns if c not in key]

    flag_cols = [c for c in ['Автоматическое', 'Аукцион', 'Органика'] if c in df.columns]
    no_sum_cols = [c for c in other if any(tok in c for tok in NO_SUM_CONTAINS)]

    # что считать «числом» для суммирования
    num_cols = []
    for c in other:
        if c in flag_cols or c in no_sum_cols:
            continue
        # аккуратно приводим потенциально числовые
        df[c] = pd.to_numeric(df[c], errors='ignore')
        if pd.api.types.is_numeric_dtype(df[c]):
            num_cols.append(c)

    text_cols = [c for c in other if c not in num_cols and c not in flag_cols and c not in no_sum_cols]

    agg = {c: 'sum' for c in num_cols}
    agg.update({c: 'max' for c in flag_cols})
    agg.update({c: 'first' for c in text_cols})
    agg.update({c: 'first' for c in no_sum_cols})  # ID и подобные — берем первое

    right_u = (df.groupby(key, as_index=False).agg(agg))

    # sanity
    assert right_u.duplicated(key).sum() == 0, "Правая часть после агрегации не уникальна по ключу"
    return right_u

def _allocate_on_left_dupes(df_merged: pd.DataFrame, spend_col: str, mode: str) -> pd.DataFrame:
    """
    Корректируем 'Расход' только в тех группах (Дата, Артикул WB), где после OUTER-merge появилось
    >1 строки из левой таблицы, чтобы суммы по датам/итогу не искажались.
    """
    out = df_merged.copy()
    out[spend_col] = pd.to_numeric(out[spend_col], errors='coerce').fillna(0.0)

    # считаем размер группы по ключу
    counts = out.groupby(KEY_COLS, dropna=False)[spend_col].transform('size')

    if mode == 'first':
        # оставляем расход только в первой строке группы; прочие — 0
        mask_first = ~out.duplicated(subset=KEY_COLS, keep='first')
        out.loc[~mask_first, spend_col] = 0.0
    elif mode == 'even':
        out[spend_col] = out[spend_col] / counts.clip(lower=1)
    else:
        raise ValueError("ALLOC_MODE должен быть 'first' или 'even'.")

    return out

# ----------- основной блок -----------
def build_funnel_expenses_outer(df_funnel_reference: pd.DataFrame,
                                df_expenses: pd.DataFrame,
                                alloc_mode: str = ALLOC_MODE) -> pd.DataFrame:
    """
    OUTER-merge: сохраняет ВСЮ сумму 'Расход, руб' по датам и в целом (как в df_expenses),
    исключает раздувание на дубликатах слева.
    """
    # 0) нормализация
    left = _norm_keys(df_funnel_reference)
    right_u = _make_right_unique(df_expenses)     # 1 строка на ключ справа

    spend_col = _pick_spend_col(right_u)
    base_by_date = (right_u.groupby('Дата', as_index=False)[spend_col]
                          .sum(min_count=1).rename(columns={spend_col: 'Расход_до'}))
    base_total = right_u[spend_col].sum()

    # 1) OUTER-merge: чтобы не потерять расходы по ключам, которых нет в воронке
    df = left.merge(right_u, on=KEY_COLS, how='outer', indicator=True)

    # 2) корректируем расход только где есть дубликаты по ключу (т.е. несколько левых строк на один ключ)
    #    Для правых «одиноких» ключей (которых не было в воронке) размер группы = 1 — значение не меняется.
    df = _allocate_on_left_dupes(df, spend_col, alloc_mode)

    # 3) проверки инвариантов по датам и общая сумма
    after_by_date = (df.groupby('Дата', as_index=False)[spend_col]
                       .sum(min_count=1).rename(columns={spend_col: 'Расход_после'}))
    chk = base_by_date.merge(after_by_date, on='Дата', how='outer').fillna(0)
    drift = chk.loc[~np.isclose(chk['Расход_до'], chk['Расход_после'], rtol=1e-9, atol=1e-6)]
    after_total = df[spend_col].sum()

    if not drift.empty or not np.isclose(after_total, base_total, rtol=1e-9, atol=1e-6):
        # покажем топ расхождений для быстрой отладки
        print("[WARN] Найдены расхождения после объединения (первые 10 дат):")
        print(drift.sort_values('Расход_до', ascending=False).head(10))
        print(f"[WARN] Общая сумма: до={base_total:,.2f} | после={after_total:,.2f}")
        # не падаем, а выводим диагностику — чтобы вы видели конкретные даты

    # 4) (опционально) можно убрать служебную колонку индикатора
    df.drop(columns=['_merge'], inplace=True, errors='ignore')

    return df

df_funnel_expenses = build_funnel_expenses_outer(df_funnel_reference, df_expenses)

# 15. Связать "ВоронкаСправочникЗатраты" с "Цены"
try:
    print("Начинаем создавать таблицу ВоронкаСправочникЗатратыЦены...")
    start_time = time.time()  # Запускаем таймер
    df_funnel_prices = pd.merge(df_funnel_expenses, df_price, on=["Дата", "Артикул"], how="left")
    df_funnel_prices = format_date_column(df_funnel_prices, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВоронкаСправочникЗатратыЦены:")
    print(df_funnel_prices.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВоронкаСправочникЗатратыЦены успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочникЗатратыЦены: {e}")

Начинаем создавать таблицу ВоронкаСправочник...


C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_9520\3488103442.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reference_filtered["Артикул WB"] = df_reference_filtered["Артикул WB"].fillna('').astype(str)


Первые 5 строк таблицы ВоронкаСправочник:
          Дата Артикул WB  Рейтинг карточки  Показы  \
0   15.12.2025    8204399               8.5  335938   
6   15.12.2025  243289986               8.5  261086   
12  15.12.2025  174535735              10.0   93133   
18  15.12.2025  174533942               9.5   55454   
24  15.12.2025  243289533               8.5  172755   

    Показы на карточке товара  Положили в корзину  Заказали, шт  Выкупили, шт  \
0                       22004                1122           452             2   
6                       10944                 793           192             0   
12                       6608                 582           162             0   
18                       3648                 389           154             0   
24                       8992                 584           151             0   

    Отменили, шт  Заказали на сумму, руб  ...  Бизнес-группа  Техсегмент  \
0              0                505303.0  ...          Обувь   f

C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_9520\3488103442.py:173: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors='ignore')


Начинаем создавать таблицу ВоронкаСправочникЗатратыЦены...
Первые 5 строк таблицы ВоронкаСправочникЗатратыЦены:
         Дата Артикул WB  Рейтинг карточки  Показы  Показы на карточке товара  \
0  01.01.2026   10005706              10.0     1.0                        1.0   
1  01.01.2026   10005707              10.0    85.0                        6.0   
2  01.01.2026   10005708               6.0     3.0                        0.0   
3  01.01.2026   10005709              10.0     2.0                        0.0   
4  01.01.2026   10005710              10.0    13.0                        0.0   

   Положили в корзину  Заказали, шт  Выкупили, шт  Отменили, шт  \
0                 0.0           0.0           0.0           0.0   
1                 2.0           0.0           0.0           0.0   
2                 0.0           0.0           0.0           0.0   
3                 0.0           0.0           0.0           0.0   
4                 0.0           0.0           0.0           0.0   

In [30]:
df_expenses

,Дата,Артикул WB,Тип активности,Автоматическое,Аукцион,Органика,Автоматическое_ID кампании,"Автоматическое_Расход, руб","Автоматическое_Рекламные Заказы, шт",Автоматическое_Рекламные в корзину,...,Аукцион_Рекламные заказов на сумму,Аукцион_Рекламные клики,Аукцион_Рекламные показы,Рекламные показы,Рекламные заказов на сумму,"Рекламные Заказы, шт",Рекламные клики,Рекламные в корзину,"Рекламные заказаных товаров, шт","Расход, руб"
0,01.01.2026,5259192,Автоматическое,1,0,0,31804656,118.0,1,2,...,0,0,0,1353,2296,1,32,2,1,118.0
1,01.01.2026,5259199,Автоматическое,1,0,0,31152108,122.0,2,13,...,0,0,0,1462,5500,2,70,13,2,122.0
2,01.01.2026,5473335,Автоматическое,1,0,0,30300233,1490.0,1,0,...,0,0,0,6410,4583,1,10,0,1,1490.0
3,01.01.2026,5594434,Автоматическое,1,0,0,31152082,49.0,2,3,...,0,0,0,632,1062,2,16,3,2,49.0
4,01.01.2026,5594442,Автоматическое,1,0,0,31203511,68.0,3,3,...,0,0,0,456,2809,3,14,3,3,68.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260332,31.12.2025,676230447,Автоматическое,1,0,0,32044223,1468.0,0,0,...,0,0,0,3372,0,0,31,0,0,1468.0
260333,31.12.2025,676230448,Автоматическое,1,0,0,32044225,715.0,0,1,...,0,0,0,1631,0,0,6,1,0,715.0
260334,31.12.2025,676230467,Автоматическое,1,0,0,31958628,624.0,0,0,...,0,0,0,1422,0,0,1,0,0,624.0
260335,31.12.2025,676230487,Автоматическое,1,0,0,31958627,672.0,0,0,...,0,0,0,1532,0,0,6,0,0,672.0


In [31]:
df_expenses[df_expenses['Дата'] == "10.11.2025"]['Расход, руб'].sum()

np.float64(0.0)

In [32]:
df_expenses['Расход, руб'].sum()

np.float64(325470053.0)

In [33]:
df_funnel_expenses[df_funnel_expenses['Дата'] == "10.11.2025"]['Расход, руб'].sum()

np.float64(0.0)

In [34]:
df_funnel_expenses['Расход, руб'].sum()

np.float64(325470053.0)

In [35]:
print(df_expenses[df_expenses['Тип активности'] == "Автоматическое"]['Расход, руб'].sum())
print(df_expenses[df_expenses['Тип активности'] == "Автоматическое"]['Автоматическое_Расход, руб'].sum())

271698065.0
271698065.0


In [36]:
print(df_funnel_expenses[df_funnel_expenses['Тип активности'] == "Автоматическое"]['Расход, руб'].sum())
print(df_funnel_expenses[df_funnel_expenses['Тип активности'] == "Автоматическое"]['Автоматическое_Расход, руб'].sum())

271698065.0
272438246.0


# STOCK WITH DISTRIBUTION

In [37]:
# 9. Получить данные таблицы с SQL (РазмерыНаАгрегаторе)
try:
    print("Начинаем получать данные для РазмеровНаАгрегаторе...")
    start_time = time.time()  # Запускаем таймер
    query_sizes = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], COUNT(DISTINCT(a.[INVENTSIZEID])) AS [Колво размеров]
        FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
        ORDER BY [dt]
        DESC
    """
    df_sizes = pd.read_sql(query_sizes, Engine)
    df_sizes = format_date_column(df_sizes, 'Дата')
    # Вывод первых 5 строк
    print("Первые 5 строк таблицы РазмерыНаАгрегаторе:")
    print(df_sizes.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для РазмеровНаАгрегаторе: {e}")

# 2. Получить данные таблицы с SQL (Остатки)
try:
    print("Начинаем получать данные для Остатков...")
    start_time = time.time()  # Запускаем таймер
    Engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)
    query_stock = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], SUM(a.qte) AS [Остаток Агрегатора]
        FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
        ORDER BY [dt]
        DESC
    """
    df_stock = pd.read_sql(query_stock, Engine)
    df_stock = format_date_column(df_stock, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатков:")
    print(df_stock.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Остатков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Остатков: {e}")

# 10. Создание таблицы "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу ВсегоРазмеров...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Артикул", "Размер"]
    for col in required_columns:
        if col not in df_reference.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_reference.")
            exit()

    # Очищаем столбец "Размер":
    # - Преобразуем в строковый формат
    # - Удаляем лишние пробелы
    # - Заменяем пустые строки на None
    df_reference["Размер"] = df_reference["Размер"].astype(str).str.strip().replace('', None)

    # Создаем DataFrame с количеством размеров для каждого артикула
    df_reference_unique = (
        df_reference
        .drop_duplicates(subset=["Артикул", "Размер"])  # Удаляем дубликаты Артикул-Размер
        .groupby("Артикул")["Размер"]  # Группируем по артикулу
        .apply(lambda sizes: len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1)  # Подсчитываем размеры
        .reset_index(name="Всего размеров")
    )

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВсегоРазмеров:")
    print(df_reference_unique.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВсегоРазмеров успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВсегоРазмеров: {e}")

# 11. Связать "РазмерыНаАгрегаторе" с "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу Дистрибуция...")
    start_time = time.time()  # Запускаем таймер

    # Объединяем таблицы по полю "Артикул"
    df_distribution = pd.merge(df_sizes, df_reference_unique, on="Артикул", how="left")

    # Вычисляем дистрибуцию с проверкой на деление на ноль
    df_distribution["Дистрибуция"] = df_distribution.apply(
        lambda row: row["Колво размеров"] / row["Всего размеров"] if row["Всего размеров"] != 0 else 0,
        axis=1
    )

    # Оставляем только нужные столбцы
    df_distribution = df_distribution[["Дата", "Артикул", "Дистрибуция"]]

    # Форматирование даты
    df_distribution = format_date_column(df_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Дистрибуция:")
    print(df_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Дистрибуция успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Дистрибуция: {e}")

# 12. Связать "Остатки" с "Дистрибуция"
try:
    print("Начинаем создавать таблицу Остатки с дистрибуцией...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution = pd.merge(df_stock, df_distribution, on=["Дата", "Артикул"], how="left")
    df_stock_with_distribution = format_date_column(df_stock_with_distribution, 'Дата')
    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатки с дистрибуцией:")
    print(df_stock_with_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Остатки с дистрибуцией успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Остатки с дистрибуцией: {e}")

Начинаем получать данные для РазмеровНаАгрегаторе...
Первые 5 строк таблицы РазмерыНаАгрегаторе:
         Дата   Артикул  Колво размеров
0  04.02.2026  00004290               1
1  04.02.2026  00004300               3
2  04.02.2026  00004510               1
3  04.02.2026  00004520               3
4  04.02.2026  00004570               1
Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: 0 часа(ов) 1 минут(ы) 7.06 секунд
Начинаем получать данные для Остатков...
Первые 5 строк таблицы Остатков:
         Дата   Артикул  Остаток Агрегатора
0  04.02.2026  00004290                   0
1  04.02.2026  00004300                   0
2  04.02.2026  00004510                   0
3  04.02.2026  00004520                   0
4  04.02.2026  00004570                   0
Данные для Остатков успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 56.32 секунд
Начинаем создавать таблицу ВсегоРазмеров...
Первые 5 строк таблицы ВсегоРазмеров:
    Артикул  Всего размеров
0  00001851         

# ДБбезПризнаков

In [38]:
# 7. Получить данные из файлов вложенной папки "Воронка с показами ВБ"
try:
    print("Начинаем получать данные для Воронки с показами ВБ...")
    start_time = time.time()  # Запускаем таймер
    folder_path_funnel_shows = os.path.join(FOLDER_PATH, "Воронка с показами ВБ")
    df_funnel_shows = pd.DataFrame()

    if os.path.exists(folder_path_funnel_shows):
        for file in os.listdir(folder_path_funnel_shows):
            file_path = os.path.join(folder_path_funnel_shows, file)

            # Пропускаем скрытые файлы
            if is_hidden(file_path):
                print(f"Пропущен скрытый файл: {file}")
                continue

            # Проверяем расширение файла
            if file.endswith((".xlsx", ".xls")):
                try:
                    # Чтение файла с явным преобразованием столбца "nm" в строку
                    if file.endswith(".xlsx"):
                        temp_df = pd.read_excel(
                            file_path,
                            sheet_name="Лист1",
                            engine="calamine",
                            usecols=range(6),
                            dtype={"nm": str}  # Явно задаем тип данных для столбца "nm"
                        )
                    elif file.endswith(".xls"):
                        temp_df = pd.read_excel(
                            file_path,
                            sheet_name="Лист1",
                            engine="xlrd",
                            usecols=range(6),
                            dtype={"nm": str}  # Явно задаем тип данных для столбца "nm"
                        )

                    # Переименование столбцов
                    temp_df.rename(columns={
                        "data_day": "Дата",
                        "nm": "Артикул WB",
                        "pokaz": "Показы из выгрузки",
                        "click": "Клики из выгрузки",
                        "basket": "Корзины из выгрузки",
                        "order": "Заказы из выгрузки"
                    }, inplace=True, errors="ignore")  # Игнорируем ошибки при переименовании

                    # Очистка данных в столбце "Артикул WB":
                    # - Удаляем лишние пробелы
                    # - Удаляем символы после точки (если есть)
                    temp_df["Артикул WB"] = temp_df["Артикул WB"].str.strip().str.split('.').str[0]

                    # Форматирование даты
                    temp_df = format_date_column(temp_df, 'Дата')

                    # Добавляем временный DataFrame к основному
                    df_funnel_shows = pd.concat([df_funnel_shows, temp_df])
                except Exception as e:
                    print(f"Ошибка при чтении файла {file}: {e}")

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Воронка с показами:")
        print(df_funnel_shows.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Воронки с показами ВБ успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Папка 'Воронка с показами ВБ' не найдена.")
except Exception as e:
    print(f"Ошибка при получении данных для Воронки с показами ВБ: {e}")

Начинаем получать данные для Воронки с показами ВБ...
Первые 5 строк таблицы Воронка с показами:
         Дата Артикул WB  Показы из выгрузки  Клики из выгрузки  \
0  04.08.2025   10336125                22.0                0.0   
1  04.08.2025  177697350               604.0               35.0   
2  04.08.2025  443740067              1091.0              163.0   
3  04.08.2025    5833879                 1.0                0.0   
4  04.08.2025   53453484               438.0               50.0   

   Корзины из выгрузки  Заказы из выгрузки  
0                  0.0                 0.0  
1                  3.0                 1.0  
2                  1.0                 0.0  
3                  0.0                 0.0  
4                  0.0                 0.0  
Данные для Воронки с показами ВБ успешно сохранены. Время выполнения: 0 часа(ов) 3 минут(ы) 57.88 секунд


In [39]:
# 8. Получить данные из файла !!!_Признаки для артикула и даты для ВБ
try:
    print("Начинаем получать данные для Признаков...")
    start_time = time.time()  # Запускаем таймер
    file_path_features = "!!!_Признаки для артикула и даты для ВБ.xlsx"
    if os.path.exists(file_path_features):
        df_item_features = pd.read_excel(file_path_features, sheet_name="Признаки для артикула", dtype=str, engine="openpyxl")
        df_date_features = pd.read_excel(file_path_features, sheet_name="Признаки для дат", dtype={0: "datetime64[ns]", **{i: str for i in range(1, 6)}}, engine="openpyxl")

        # Обработка ошибок
        df_item_features = handle_errors(df_item_features)
        df_date_features = handle_errors(df_date_features)

        # Форматирование даты
        df_date_features = format_date_column(df_date_features, 'Дата')

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки артикула:")
        print(df_item_features.head())

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки дат:")
        print(df_date_features.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Признаков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл '!!!_Признаки для артикула и даты для ВБ.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Признаков: {e}")

Начинаем получать данные для Признаков...
Первые 5 строк таблицы Признаки артикула:
    Артикул Признак Артикула 1 Признак Артикула 2 Признак Артикула 3  \
0  00001851                NaN                NaN                NaN   
1  00001852                NaN                NaN                NaN   
2  00001855                NaN                NaN                NaN   
3  00001856                NaN                NaN                NaN   
4  00001931                NaN                NaN                NaN   

   Признак Артикула 4  Признак Артикула 5  
0                 NaN                 NaN  
1                 NaN                 NaN  
2                 NaN                 NaN  
3                 NaN                 NaN  
4                 NaN                 NaN  
Первые 5 строк таблицы Признаки дат:
Empty DataFrame
Columns: [Дата, Признак Даты 1, Признак Даты 2, Признак Даты 3, Признак Даты 4, Признак Даты 5]
Index: []
Данные для Признаков успешно сохранены. Время выполнения: 0 

In [40]:
# 16. Связать "ВоронкаСправочникЗатратыЦены" с "Остатки с дистрибуцией"
try:
    print("Начинаем создавать таблицу ДБбезПризнаков...")
    start_time = time.time()  # Запускаем таймер
    df_final_db = pd.merge(df_funnel_prices, df_stock_with_distribution, on=["Дата", "Артикул"], how="left")
    df_final_db = format_date_column(df_final_db, 'Дата')
    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБбезПризнаков:")
    print(df_final_db.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБбезПризнаков успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБбезПризнаков: {e}")
del df_funnel_prices
# 16.5 Связать "ДБбезПризнаков" с "Воронка с показами ВБ"
try:
    print("Начинаем создавать таблицу ДБсПоказами...")
    start_time = time.time()  # Запускаем таймер

    # Проверяем наличие необходимых столбцов в df_funnel_shows
    required_columns = ["Дата", "Артикул WB"]
    for col in required_columns:
        if col not in df_funnel_shows.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_funnel_shows.")
            exit()

    # Приводим типы данных к строковому формату
    df_final_db["Артикул WB"] = (df_final_db["Артикул WB"]
        .fillna('')
        .astype(str)
        .str.strip()
        .str.upper()
    )
    df_funnel_shows["Артикул WB"] = (df_funnel_shows["Артикул WB"]
        .fillna('')
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Объединяем таблицы по полям "Дата" и "Артикул WB"
    df_final_db_with_shows = pd.merge(
        df_final_db,
        df_funnel_shows,
        left_on=["Дата", "Артикул WB"],
        right_on=["Дата", "Артикул WB"],
        how="left"
    )

    # Переименовываем столбец "Дата" (если нужно)
    #df_final_db_with_shows = df_final_db_with_shows.drop(columns=["Дата"], errors="ignore")

    # Форматирование даты
    df_final_db_with_shows = format_date_column(df_final_db_with_shows, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПоказами:")
    print(df_final_db_with_shows.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПоказами успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПоказами: {e}")

# 17. Связать "ДБсПоказами" с "Признаки для артикула"
try:
    print("Начинаем создавать таблицу ДБсПризнакамиАртикула...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_item_features = pd.merge(df_final_db_with_shows, df_item_features, on="Артикул", how="left")
    df_final_db_item_features = format_date_column(df_final_db_item_features, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнакамиАртикула:")
    print(df_final_db_item_features.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнакамиАртикула успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнакамиАртикула: {e}")

# 18. Связать "ДБсПризнакамиАртикула" с "Признаки для дат"
try:
    print("Начинаем создавать таблицу ДБсПризнаками...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_all_features = pd.merge(df_final_db_item_features, df_date_features, on="Дата", how="left")
    df_final_db_all_features = format_date_column(df_final_db_all_features, 'Дата')


    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнаками:")
    print(df_final_db_all_features.head())
    
    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнаками успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнаками: {e}")

Начинаем создавать таблицу ДБбезПризнаков...
Первые 5 строк таблицы ДБбезПризнаков:
         Дата Артикул WB  Рейтинг карточки  Показы  Показы на карточке товара  \
0  01.01.2026   10005706              10.0     1.0                        1.0   
1  01.01.2026   10005707              10.0    85.0                        6.0   
2  01.01.2026   10005708               6.0     3.0                        0.0   
3  01.01.2026   10005709              10.0     2.0                        0.0   
4  01.01.2026   10005710              10.0    13.0                        0.0   

   Положили в корзину  Заказали, шт  Выкупили, шт  Отменили, шт  \
0                 0.0           0.0           0.0           0.0   
1                 2.0           0.0           0.0           0.0   
2                 0.0           0.0           0.0           0.0   
3                 0.0           0.0           0.0           0.0   
4                 0.0           0.0           0.0           0.0   

   Заказали на сумму, руб 

In [41]:
# === SQL СЦЕПКИ ВБ ===
sql = """
SELECT [updated_at] as [Дата Обновления], 
		[nmID] as [Артикул WB], 
		[imtID] as [Текущая склейка]
FROM [DBReport].[mp].[wb_scepka]
"""
df_links = pd.read_sql(sql, Engine)
df_links['Артикул WB'] = df_links['Артикул WB'].astype(str)
df_links.to_excel(os.path.join(FOLDER_PATH, f"Склейки товаров\\WB\\{df_links['Дата Обновления'].iloc[0].strftime('%d.%m.%Y')}_Склейка Товаров_WB.xlsx"))

In [42]:
df_links['Текущая склейка'].unique()

array([  6420365, 788797079, 496452162, ..., 696221267, 696217944,
       440806383], shape=(43713,))

In [43]:
df_final_db_all_features = pd.merge(df_final_db_all_features, df_links[["Артикул WB", "Текущая склейка"]], how='left', on='Артикул WB')

In [44]:
print(list(df_final_db_all_features.columns))

['Дата', 'Артикул WB', 'Рейтинг карточки', 'Показы', 'Показы на карточке товара', 'Положили в корзину', 'Заказали, шт', 'Выкупили, шт', 'Отменили, шт', 'Заказали на сумму, руб', 'Выкупили на сумму, руб', 'Отменили на сумму, руб', 'Средняя цена, руб', 'Рейтинг по отзывам', 'Артикул', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа ВБ', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'Автоматическое_Расход, руб', 'Автоматическое_Рекламные Заказы, шт', 'Автоматическое_Рекламные в корзину', 'Автоматическое_Рекламные заказаных товаров, шт', 'Автоматическое_Рекламные заказов на сумму', 'Автоматическое_Рекламные клики', 'Автоматическое_Рекламные показы', 'Аукцион_Расход, руб', 'Аукцион_Рекламные Заказы, шт', 'Аукцион_Рекламные в корзину', 'Аукцион_Рекламные заказаных товаров, шт', 'Аукцион_Рекламные заказов н

In [45]:
df_final_db_all_features['Автоматическое_ID кампании'].unique

<bound method Series.unique of 0          <NA>
1          <NA>
2          <NA>
3          <NA>
4          <NA>
           ... 
3193564    <NA>
3193565    <NA>
3193566    <NA>
3193567    <NA>
3193568    <NA>
Name: Автоматическое_ID кампании, Length: 3193569, dtype: Int64>

In [46]:
# === SQL АССОЦИАЦИИ ВБ ===
sql = """
SELECT [Дата]
      ,[nmId]
      ,[ID кампании]
      ,[Кол-во заказаных товаров, шт]
      ,[Заказов на сумму]
  FROM [DBReport].[mp].[wb_marketing_associated]
"""
df_associations = pd.read_sql(sql, Engine)

In [47]:
df_associations

,Дата,nmId,ID кампании,"Кол-во заказаных товаров, шт",Заказов на сумму
0,2025-10-03,443739680,29129782,0,0
1,2025-10-03,448976782,29129782,0,0
2,2025-10-03,443739708,29129782,0,0
3,2025-10-03,343044552,29129782,0,0
4,2025-10-03,449173099,29129782,0,0
...,...,...,...,...,...
12015670,2026-02-05,446312621,30403652,0,0
12015671,2026-02-03,334774788,33252650,0,0
12015672,2026-02-03,246492003,30014133,1,6539
12015673,2026-02-05,204158962,30502276,0,0


In [48]:
df_final_db_all_features

,Дата,Артикул WB,Рейтинг карточки,Показы,Показы на карточке товара,Положили в корзину,"Заказали, шт","Выкупили, шт","Отменили, шт","Заказали на сумму, руб",...,Признак Артикула 2,Признак Артикула 3,Признак Артикула 4,Признак Артикула 5,Признак Даты 1,Признак Даты 2,Признак Даты 3,Признак Даты 4,Признак Даты 5,Текущая склейка
0,01.01.2026,10005706,10.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,468745415.0
1,01.01.2026,10005707,10.0,85.0,6.0,2.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,468745497.0
2,01.01.2026,10005708,6.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,468745729.0
3,01.01.2026,10005709,10.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,468746003.0
4,01.01.2026,10005710,10.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,499816198.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3193564,31.12.2025,9890971,9.5,566.0,22.0,0.0,1.0,1.0,0.0,5451.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,794262955.0
3193565,31.12.2025,9890972,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,496226223.0
3193566,31.12.2025,9890973,10.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,496226230.0
3193567,31.12.2025,9933006,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7726362.0


In [45]:
import numpy as np
import pandas as pd

DDMMYYYY = re.compile(r'^\d{2}\.\d{2}\.\d{4}$')   # 01.09.2025
ISO      = re.compile(r'^\d{4}-\d{2}-\d{2}$')     # 2025-09-01

def normalize_date_strict(s: pd.Series) -> pd.Series:
    """Строго нормализует дату к datetime64[ns] без времени.
       Поддерживает РОВНО 'DD.MM.YYYY' и 'YYYY-MM-DD'. Никаких эвристик."""
    ss = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=ss.index, dtype='datetime64[ns]')

    m_dd = ss.str.match(DDMMYYYY)
    m_iso = ss.str.match(ISO)

    out[m_dd]  = pd.to_datetime(ss[m_dd],  format='%d.%m.%Y', errors='coerce')
    out[m_iso] = pd.to_datetime(ss[m_iso], format='%Y-%m-%d', errors='coerce')

    # диагностируем «левые» значения
    bad = ~(m_dd | m_iso)
    if bad.any():
        print("ВНИМАНИЕ: найдены строки с недопустимым форматом даты, примеры:")
        print(ss[bad].head(10).to_list())

    return out.dt.normalize()

def allocate_campaign_metrics_rowwise_intsafe(
    df_associations: pd.DataFrame,      # Дата, ID кампании, Кол-во заказаных товаров, Заказов на сумму
    df_item_features: pd.DataFrame,     # Дата, Артикул, Показы (или Показы, всего), Автоматическое_ID кампании / Аукцион_ID кампании
    shows_cols=('Показы, всего','Показы'),
    auto_col='Автоматическое_ID кампании',
    auction_col='Аукцион_ID кампании',
    article_col='Артикул',
    money_decimals=2
):
    A = df_associations.copy()
    I = df_item_features.copy()

    # 1) Даты
    # for c in ['Дата']:
    #     if c in A.columns: A[c] = pd.to_datetime(A[c], errors='coerce')
    #     if c in I.columns: I[c] = pd.to_datetime(I[c], errors='coerce')

    A['Дата'] = normalize_date_strict(A['Дата'])
    I['Дата'] = normalize_date_strict(I['Дата'])

    # 2) coalesce ID кампании
    I['ID кампании'] = pd.Series(index=I.index, dtype=object)
    if auto_col in I.columns:
        I['ID кампании'] = I[auto_col]
    if auction_col in I.columns:
        I['ID кампании'] = I['ID кампании'].combine_first(I[auction_col])

    # 3) колонка показов
    shows_col = next((c for c in shows_cols if c in I.columns), None)
    if shows_col is None:
        raise ValueError(f'Не найдена колонка показов ни из {shows_cols}')

    # 4) агрегируем кампанию по (Дата, ID кампании)
    A2 = (A.groupby(['Дата','ID кампании'], dropna=False)[
            ['Кол-во заказаных товаров, шт','Заказов на сумму']
         ].sum()
         .reset_index())

    # 5) LEFT merge — сохраним исходный индекс, чтобы потом ровно join'ить обратно
    keepI = ['Дата', article_col, 'ID кампании', shows_col]
    I_view = I.loc[:, [c for c in keepI if c in I.columns]].copy()
    I_view['_orig_index'] = I_view.index

    J = I_view.merge(A2, on=['Дата','ID кампании'], how='left')
    # Выравниваем индекс J обратно к исходному для безопасного .join
    J.index = I_view['_orig_index'].values

    # 6) типы
    J[shows_col] = pd.to_numeric(J[shows_col], errors='coerce').fillna(0.0)
    for m in ['Кол-во заказаных товаров, шт', 'Заказов на сумму']:
        if m not in J.columns:
            J[m] = 0.0
        J[m] = pd.to_numeric(J[m], errors='coerce').fillna(0.0)

    # 7) групповые итоги и доля
    keys = ['Дата','ID кампании']
    grp_total_shows = J.groupby(keys, dropna=False)[shows_col].transform('sum')  # это нормально: суммируем показы
    grp_total_qty   = J.groupby(keys, dropna=False)['Кол-во заказаных товаров, шт'].transform('first')
    grp_total_amt   = J.groupby(keys, dropna=False)['Заказов на сумму'].transform('first')


    m_has_total = (grp_total_qty > 0) | (grp_total_amt > 0)
    grp_size    = J.groupby(keys, dropna=False)[shows_col].transform('size').astype('float64')

    J['_share'] = 0.0
    m_shows = (grp_total_shows > 0) & m_has_total
    J.loc[m_shows, '_share'] = (J.loc[m_shows, shows_col] / grp_total_shows[m_shows])

    m_equal = (grp_total_shows == 0) & m_has_total
    denom = grp_size.where(grp_size > 0, 1.0)
    J.loc[m_equal, '_share'] = 1.0 / denom[m_equal]

    # ---------- ШТУКИ: Hamilton ----------
    J['__grp_total_qty'] = grp_total_qty
    J['__raw_qty'] = J['_share'] * J['__grp_total_qty']

    def _distribute_int(group, raw_col, total_col, out_name):
        raw = group[raw_col].to_numpy(dtype=float)
        base = np.floor(raw).astype(np.int64)
        need = int(round(group[total_col].iloc[0] - base.sum()))
        if need > 0:
            frac = raw - base
            order = np.argsort(-frac, kind='mergesort')
            base[order[:need]] += 1
        return pd.Series(base, index=group.index, name=out_name, dtype='int64')

    J['Ассоциированные заказы, шт'] = 0
    J.loc[m_has_total, 'Ассоциированные заказы, шт'] = (
        J[m_has_total]
          .groupby(keys, group_keys=False, dropna=False)
          .apply(_distribute_int, raw_col='__raw_qty', total_col='__grp_total_qty',
                 out_name='Ассоциированные заказы, шт')
    )

    # ---------- ДЕНЬГИ: в копейках, Hamilton ----------
    cents = 10 ** money_decimals
    J['__grp_total_cents'] = np.round(grp_total_amt * cents).astype('int64')
    J['__raw_cents'] = J['_share'] * J['__grp_total_cents']

    def _distribute_cents(group):
        raw = group['__raw_cents'].to_numpy(dtype=float)
        base = np.floor(raw).astype(np.int64)
        need = int(group['__grp_total_cents'].iloc[0] - base.sum())
        if need > 0:
            frac = raw - base
            order = np.argsort(-frac, kind='mergesort')
            base[order[:need]] += 1
        return pd.Series(base, index=group.index, name='__alloc_cents', dtype='int64')

    J['__alloc_cents'] = 0
    J.loc[m_has_total, '__alloc_cents'] = (
        J[m_has_total]
          .groupby(keys, group_keys=False, dropna=False)
          .apply(_distribute_cents)
    )
    J['Ассоциированные заказы, руб'] = (J['__alloc_cents'] / cents).astype('float64')

    # Нули там, где нечего распределять
    J.loc[~m_has_total, ['Ассоциированные заказы, шт','Ассоциированные заказы, руб']] = 0

    # очистка временных колонок
    J.drop(columns=[
        '_share','__grp_total_qty','__raw_qty','__grp_total_cents','__raw_cents','__alloc_cents'
    ], inplace=True, errors='ignore')

    # 8) Вернуть в исходный df_item_features 1:1 по индексу
    out = I.copy()
    out = out.join(J[['Ассоциированные заказы, шт','Ассоциированные заказы, руб']], how='left')
    out[['Ассоциированные заказы, шт','Ассоциированные заказы, руб']] = \
        out[['Ассоциированные заказы, шт','Ассоциированные заказы, руб']].fillna(0)

    assert len(out) == len(df_item_features), f"Row count changed: {len(out)} vs {len(df_item_features)}"
    out['Дата'] = pd.to_datetime(out['Дата'], errors='coerce')\
                 .dt.strftime('%d.%m.%Y')
    return out

df_alloc = allocate_campaign_metrics_rowwise_intsafe(df_associations, df_final_db_all_features)

C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_15912\2372522188.py:115: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  J[m_has_total]
C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_15912\2372522188.py:140: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_distribute_cents)


In [46]:
df_alloc['Дата'].unique()

array(['01.01.2026', '01.02.2026', '02.01.2026', '02.02.2026',
       '03.01.2026', '03.02.2026', '04.01.2026', '04.02.2026',
       '05.01.2026', '06.01.2026', '07.01.2026', '08.01.2026',
       '09.01.2026', '10.01.2026', '11.01.2026', '12.01.2026',
       '13.01.2026', '14.01.2026', '15.01.2026', '15.12.2025',
       '16.01.2026', '16.12.2025', '17.01.2026', '17.12.2025',
       '18.01.2026', '18.12.2025', '19.01.2026', '19.12.2025',
       '20.01.2026', '20.12.2025', '21.01.2026', '21.12.2025',
       '22.01.2026', '22.12.2025', '23.01.2026', '23.12.2025',
       '24.01.2026', '24.12.2025', '25.01.2026', '25.12.2025',
       '26.01.2026', '26.12.2025', '27.01.2026', '27.12.2025',
       '28.01.2026', '28.12.2025', '29.01.2026', '29.12.2025',
       '30.01.2026', '30.12.2025', '31.01.2026', '31.12.2025'],
      dtype=object)

In [47]:
wide[wide['Дата'] == "10.11.2025"]['Расход, руб'].sum()

np.float64(0.0)

In [48]:
df_final_db_all_features[df_final_db_all_features['Дата'] == "10.11.2025"]['Показы'].sum()

np.float64(0.0)

In [49]:
rub = df_alloc['Ассоциированные заказы, руб']
# проверяем, что все кратны 1 копейке
mask_not_cents = (np.round(rub * 100) != rub * 100)

df_not_cents = df_alloc[mask_not_cents]
print(len(df_not_cents))
print(df_not_cents[['Дата', 'ID кампании', 'Артикул', 'Ассоциированные заказы, руб']].head(10))


14
               Дата  ID кампании   Артикул  Ассоциированные заказы, руб
1429284  17.12.2025     31886211  a7409010                      4476.06
1806799  20.12.2025     31886437  a7409070                     34307.95
1933052  21.12.2025     31886211  a7409000                      8848.87
1933053  21.12.2025     31886211  a7409020                      9102.04
2184739  23.12.2025     31886211  a7409010                     19418.17
2436955  25.12.2025     31886211  a7409020                     20622.40
2562999  26.12.2025     31886437  a7409030                     40427.02
2563007  26.12.2025     31886437  a7409050                     39268.98
2688377  27.12.2025     31886437  a7409030                     36930.37
2688385  27.12.2025     31886437  a7409050                     37111.63


In [50]:
print(df_alloc[['Ассоциированные заказы, шт', 'Ассоциированные заказы, руб']].dtypes)

Ассоциированные заказы, шт       int64
Ассоциированные заказы, руб    float64
dtype: object


In [51]:
df_alloc[df_alloc['Дата'] == "10.11.2025"]['Показы'].sum()

np.float64(0.0)

In [52]:
df_alloc.columns

Index(['Дата', 'Артикул WB', 'Рейтинг карточки', 'Показы',
       'Показы на карточке товара', 'Положили в корзину', 'Заказали, шт',
       'Выкупили, шт', 'Отменили, шт', 'Заказали на сумму, руб',
       'Выкупили на сумму, руб', 'Отменили на сумму, руб', 'Средняя цена, руб',
       'Рейтинг по отзывам', 'Артикул', 'Наименование', 'Коллекция', 'Бренд',
       'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа',
       'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции',
       'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа ВБ', 'НДС',
       'Ответственный за группу', 'Группа для отчетов',
       'Автоматическое_Расход, руб', 'Автоматическое_Рекламные Заказы, шт',
       'Автоматическое_Рекламные в корзину',
       'Автоматическое_Рекламные заказаных товаров, шт',
       'Автоматическое_Рекламные заказов на сумму',
       'Автоматическое_Рекламные клики', 'Автоматическое_Рекламные показы',
       'Аукцион_Расход, руб', 'Аукцион_Рекламные Заказы, шт'

In [53]:
print(df_alloc[df_alloc['Тип активности'] == "Автоматическое"]['Расход, руб'].sum())
print(df_alloc[df_alloc['Тип активности'] == "Аукцион"]['Расход, руб'].sum())
print(df_alloc[df_alloc['Тип активности'] == "Автоматическое и Аукцион"]['Расход, руб'].sum())
print(df_alloc[df_alloc['Тип активности'] == "Органика"]['Расход, руб'].sum())

268768540.0
44708806.0
8133238.0
0.0


In [54]:
print(wide['Расход, руб'].sum())

321610584.0


In [55]:
df_alloc['Дата'].unique()

array(['01.01.2026', '01.02.2026', '02.01.2026', '02.02.2026',
       '03.01.2026', '03.02.2026', '04.01.2026', '04.02.2026',
       '05.01.2026', '06.01.2026', '07.01.2026', '08.01.2026',
       '09.01.2026', '10.01.2026', '11.01.2026', '12.01.2026',
       '13.01.2026', '14.01.2026', '15.01.2026', '15.12.2025',
       '16.01.2026', '16.12.2025', '17.01.2026', '17.12.2025',
       '18.01.2026', '18.12.2025', '19.01.2026', '19.12.2025',
       '20.01.2026', '20.12.2025', '21.01.2026', '21.12.2025',
       '22.01.2026', '22.12.2025', '23.01.2026', '23.12.2025',
       '24.01.2026', '24.12.2025', '25.01.2026', '25.12.2025',
       '26.01.2026', '26.12.2025', '27.01.2026', '27.12.2025',
       '28.01.2026', '28.12.2025', '29.01.2026', '29.12.2025',
       '30.01.2026', '30.12.2025', '31.01.2026', '31.12.2025'],
      dtype=object)

In [56]:
df_alloc['Ассоциированные заказы, шт'].unique()

array([  0,   4,   2,   1,   3,   5,   7,  16,   9,   8,  11,   6,  14,
        19,  12,  13,  64,  17,  52,  20,  10,  47,  29,  53,  28,  32,
        21,  15,  35,  33,  18,  34,  23,  37,  25,  50,  43,  93,  22,
        26,  27,  59,  24,  40,  39,  41,  54,  45,  55,  48,  46,  71,
        36,  62,  30,  60,  57,  42, 128,  44,  31,  73,  68, 106,  51,
        66,  58,  38,  87,  69, 124,  82,  80, 111,  72,  61,  81,  49,
        96, 127,  76,  77, 112,  63,  86,  65,  88,  70, 131, 100, 103,
        95,  79, 218,  56, 132, 114, 102, 153,  75,  83,  85,  74,  67,
       104, 126,  94,  91, 101, 135, 121,  90, 143, 116, 105, 115,  78,
       125,  98,  92, 108, 120, 118,  89, 110,  97, 154, 148, 130, 109,
       275, 287,  84, 156, 169, 107, 166, 145, 142, 134, 147,  99, 129,
       123, 117, 133, 122, 113, 159, 146, 182, 188, 185, 144, 162, 157,
       192, 136, 155, 150, 160, 152, 171, 119, 139])

In [57]:
len(df_alloc.columns)

81

In [58]:
df_alloc[df_alloc['Дата']=="01.10.2025"].to_csv('csv.csv')

In [59]:
df_alloc['Ассоциированные заказы, шт'].unique()

array([  0,   4,   2,   1,   3,   5,   7,  16,   9,   8,  11,   6,  14,
        19,  12,  13,  64,  17,  52,  20,  10,  47,  29,  53,  28,  32,
        21,  15,  35,  33,  18,  34,  23,  37,  25,  50,  43,  93,  22,
        26,  27,  59,  24,  40,  39,  41,  54,  45,  55,  48,  46,  71,
        36,  62,  30,  60,  57,  42, 128,  44,  31,  73,  68, 106,  51,
        66,  58,  38,  87,  69, 124,  82,  80, 111,  72,  61,  81,  49,
        96, 127,  76,  77, 112,  63,  86,  65,  88,  70, 131, 100, 103,
        95,  79, 218,  56, 132, 114, 102, 153,  75,  83,  85,  74,  67,
       104, 126,  94,  91, 101, 135, 121,  90, 143, 116, 105, 115,  78,
       125,  98,  92, 108, 120, 118,  89, 110,  97, 154, 148, 130, 109,
       275, 287,  84, 156, 169, 107, 166, 145, 142, 134, 147,  99, 129,
       123, 117, 133, 122, 113, 159, 146, 182, 188, 185, 144, 162, 157,
       192, 136, 155, 150, 160, 152, 171, 119, 139])

In [60]:
df_alloc['Ассоциированные заказы, руб'].unique()

array([    0.,  2900.,  5248., ..., 18636., 89611., 22742.],
      shape=(44099,))

In [61]:
df_alloc['Ассоциированные заказы, руб'] = pd.to_numeric(df_alloc['Ассоциированные заказы, руб'])

In [62]:
df_alloc

,Дата,Артикул WB,Рейтинг карточки,Показы,Показы на карточке товара,Положили в корзину,"Заказали, шт","Выкупили, шт","Отменили, шт","Заказали на сумму, руб",...,Признак Артикула 5,Признак Даты 1,Признак Даты 2,Признак Даты 3,Признак Даты 4,Признак Даты 5,Текущая склейка,ID кампании,"Ассоциированные заказы, шт","Ассоциированные заказы, руб"
0,01.01.2026,10005706,10.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,468745415.0,<NA>,0,0.0
1,01.01.2026,10005707,10.0,85.0,6.0,2.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,468745497.0,<NA>,0,0.0
2,01.01.2026,10005708,6.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,468745729.0,<NA>,0,0.0
3,01.01.2026,10005709,10.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,468746003.0,<NA>,0,0.0
4,01.01.2026,10005710,10.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,499816198.0,<NA>,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3193564,31.12.2025,9890971,9.5,566.0,22.0,0.0,1.0,1.0,0.0,5451.0,...,NaN,NaN,NaN,NaN,NaN,NaN,794262955.0,<NA>,0,0.0
3193565,31.12.2025,9890972,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,496226223.0,<NA>,0,0.0
3193566,31.12.2025,9890973,10.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,496226230.0,<NA>,0,0.0
3193567,31.12.2025,9933006,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,7726362.0,<NA>,0,0.0


In [63]:
import pyarrow as pa
import pyarrow.csv as csv
table = pa.Table.from_pandas(df_alloc)

In [64]:
csv.write_csv(table, os.path.join(FOLDER_PATH, "ДБсПризнаками.csv"))

In [65]:
# df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками.csv"), index=False)

In [66]:
# Функция для обновления Excel-файла с циклом попыток
def update_and_save_excel(file_path, new_file_path):
    max_attempts = 10  # Максимальное количество попыток
    attempt = 0 

    while attempt < max_attempts:
        attempt += 1
        print(f"Попытка {attempt} обновить файл '{os.path.basename(file_path)}'...")

        try:
            # Открываем Excel приложение
            excel = win32.Dispatch("Excel.Application")
            excel.DisplayAlerts = False  # Отключает предупреждения Excel

            try:
                # Открываем книгу
                workbook = excel.Workbooks.Open(file_path)

                # Выполняем обновление всех данных (эквивалентно "Обновить всё" в Excel)
                print("Выполняем обновление данных...")
                workbook.RefreshAll()
                excel.CalculateUntilAsyncQueriesDone()  # Дожидаемся завершения обновления

                # Сохраняем оригинальный файл в FOLDER_PATH_FOR_DB
                workbook.SaveAs(file_path)
                print(f"Файл успешно сохранен с оригинальным именем в '{os.path.dirname(file_path)}'.")

                # Сохраняем файл с новым именем в FOLDER_PATH_FEATURES
                workbook.SaveAs(new_file_path)
                print(f"Файл успешно сохранен как '{os.path.basename(new_file_path)}'.")

                return True  # Успешное завершение

            except Exception as e:
                print(f"Ошибка при обновлении или сохранении файла: {e}")
            finally:
                # Закрываем книгу и выходим из Excel
                if 'workbook' in locals():
                    workbook.Close(SaveChanges=False)
                excel.Quit()

        except Exception as e:
            print(f"Ошибка при работе с Excel: {e}")

        # Если произошла ошибка, ждем перед следующей попыткой
        if attempt < max_attempts:
            print(f"Пауза перед следующей попыткой ({attempt + 1}/{max_attempts})...")
            time.sleep(60)  # Пауза 5 секундS

    return False  # Все попытки завершились неудачно

In [67]:
# 19. Обновить файл "Показы и затраты ОЗ_2.0.xlsx"
try:
    print("Подготовка данных для ДБ завершена.")
    # input("Начать обновление файлов ДБ? Для подтверждения нажмите Enter...")
    print("Начинаем обновлять файл 'Показы и затраты ВБ_2.0.xlsx'...")
    start_time = time.time()  # Запускаем таймер

    # Путь к исходному файлу
    file_path_shows_expenses = os.path.join(FOLDER_PATH_FOR_DB, "Показы и затраты ВБ_2.0.xlsx")

    if os.path.exists(file_path_shows_expenses):
        # Создаем новое имя файла с текущей датой без года
        current_month_day = time.strftime("%d.%m")  # Текущая дата в формате ДД.ММ
        new_file_name = f"Показы и затраты ВБ_2.0 {current_month_day}.xlsx"
        new_file_path = os.path.join(FOLDER_PATH_FEATURES, new_file_name)

        # Путь для сохранения в дополнительную папку FOLDER_PATH_DUDL
        dudl_file_path = os.path.join(FOLDER_PATH_DUDL, new_file_name)

        # Удаляем старые файлы из FOLDER_PATH_DUDL
        try:
            if os.path.exists(FOLDER_PATH_DUDL):
                for filename in os.listdir(FOLDER_PATH_DUDL):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ВБ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_DUDL, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_DUDL}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_DUDL}': {delete_error}")

        # Удаляем старые файлы из FOLDER_PATH_FEATURES
        try:
            if os.path.exists(FOLDER_PATH_FEATURES):
                for filename in os.listdir(FOLDER_PATH_FEATURES):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ВБ_2\.0 (\d{2}\.\d{2})\.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_FEATURES, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_FEATURES}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_FEATURES}': {delete_error}")

        # Пытаемся обновить и сохранить файл
        success = update_and_save_excel(file_path_shows_expenses, new_file_path)
        # success = True

        if not success:
            # Если все попытки неудачны, выводим сообщение пользователю
            while not success:
                input("Обновить Excel файл не получилось. Закройте все открытые файлы и нажмите любую кнопку для повторной попытки.")
                success = update_and_save_excel(file_path_shows_expenses, new_file_path)

            print("Файл успешно обновлен после повторной попытки.")

        # После успешного обновления копируем файл в папку FOLDER_PATH_DUDL
        if success:
            try:
                shutil.copy(new_file_path, dudl_file_path)
                print(f"Файл успешно скопирован в папку '{FOLDER_PATH_DUDL}'.")
            except Exception as copy_error:
                print(f"Ошибка при копировании файла в папку '{FOLDER_PATH_DUDL}': {copy_error}")

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Файл успешно обновлен и сохранен. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Показы и затраты ВБ_2.0.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при обработке файла 'Показы и затраты ВБ_2.0.xlsx': {e}")

Подготовка данных для ДБ завершена.
Начинаем обновлять файл 'Показы и затраты ВБ_2.0.xlsx'...
Файл 'Показы и затраты ВБ_2.0 03.02.xlsx' удален из папки '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл 'Показы и затраты ВБ_2.0 03.02.xlsx' удален из папки '\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям'.
Попытка 1 обновить файл 'Показы и затраты ВБ_2.0.xlsx'...
Выполняем обновление данных...
Файл успешно сохранен с оригинальным именем в '\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям'.
Файл успешно сохранен как 'Показы и затраты ВБ_2.0 04.02.xlsx'.
Файл успешно скопирован в папку '\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ'.
Файл успешно обновлен и сохранен. Время выполнения: 0 часа(ов) 17 минут(ы) 2.16 секунд
